# Libs

In [ ]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import time
import gc
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

import futurai_ppd as ppd
import futurai_utils as utils
from futurai_ml_dev import FuturaiML

def set_deterministic(seed=42):
    # Python nativo
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # NumPy
    np.random.seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # Para multi-GPU
    
    # Garantir algoritmos determinísticos no back-end (CuDNN)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Executar a função
set_deterministic(42)

DIR_DATA = os.getcwd() + "/data/"

# Load Data

In [ ]:
process_name = "Compressor GARO"
timestamp="TIMESTAMP"

list_colomuns_drop = ['ARA-384R607M.PV', 'ARA-381P013B2.OP','ARA-384F934.PV','ARA-381I001H.PV','ARA-381I002H.PV','ARA-3385-FIC-522/PID1/PV.CV','ARA-3385-FI603/AI1/PV.CV','ARA-385F718.PV','ARA-381P013B.OP']

df_dataset = ppd.load_dataset_principal(DIR_DATA+process_name+".csv", list_colomuns_drop, timestamp, dropna=True, use_chunks=True, chunksize=10000)
df_dataset

# Set ON/OFF var

In [ ]:
pre_process = []
pp_var_ref_desligado = "ARA-384I607.PV"
pp_valor_ref_desligado = 30  
pp_tempo_ref_desligado = 0
pp_pre_corte_transitorio = 60 
pp_pos_corte_transitorio = 140
pre_process.append(  
{
   "after_cut": pp_pos_corte_transitorio,
   "interval_off": pp_tempo_ref_desligado,
   "limit_off": pp_valor_ref_desligado,
   "pre_cut": pp_pre_corte_transitorio,
   "variable_off": pp_var_ref_desligado
  })

# TAGs and descriptions

In [ ]:
df_sistema, df_sistema_drop =  ppd.set_tags_config(df_dataset,DIR_DATA+process_name+"_subsistema.csv")
df_sistema

# Select training periods (Atual)

In [ ]:
fig = utils.select_training_period(df_dataset, timestamp)
fig.show()

# Data Train

## Period 1

In [ ]:
# train 1
start_date_train = pd.to_datetime('2025-02-20 18:00:00')
end_date_train = pd.to_datetime('2025-02-22 00:00:00')

mask = (df_dataset[timestamp] >= start_date_train) & (df_dataset[timestamp] <= end_date_train)
df_train = df_dataset.loc[mask]

# pre - processamento
print(df_train.shape)

list_periods_train = []
for pro in pre_process:
    df_train,_,list_aux = ppd.drop_transitorio_desligado(df_train,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train.reset_index(inplace=True, drop=True)

print(df_train.shape)

eixoX_train = df_train[timestamp]
df_train = df_train.drop(timestamp,axis=1)
df_train1 = df_train.copy()


start_date_train2 = None
end_date_train2 = None

start_date_train3 = None
end_date_train3 = None

## Period 2

In [ ]:
#Data train 
start_date_train2 = pd.to_datetime('2025-01-30 00:00:00')
end_date_train2 = pd.to_datetime('2025-02-02 00:00:00')


mask = (df_dataset[timestamp] > start_date_train2) & (df_dataset[timestamp] <= end_date_train2)
df_train2 = df_dataset.loc[mask]

# pre - processamento
print(df_train2.shape)

list_periods_train = []
for pro in pre_process:
    df_train2,_,list_aux = ppd.drop_transitorio_desligado(df_train2,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train2.reset_index(inplace=True, drop=True)

print(df_train2.shape)


eixoX_train2 = df_train2[timestamp]
df_train2 = df_train2.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train2], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train2], ignore_index=True)

## Period 3

In [ ]:
#Data train 
start_date_train3 = pd.to_datetime('2025-03-06 04:30:00') 
end_date_train3 = pd.to_datetime('2025-03-06 05:30:00')

for i in range(10):
    mask = (df_dataset[timestamp] > start_date_train3) & (df_dataset[timestamp] <= end_date_train3)
    df_train3 = df_dataset.loc[mask]

    # pre - processamento
    #print(df_trainN.shape)
    list_periods_train = []
    for pro in pre_process:
        df_train3,_,list_aux = ppd.drop_transitorio_desligado(df_train3,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
        list_periods_train = [*list_periods_train,*list_aux]

    df_train3.reset_index(inplace=True, drop=True)

    #print(df_trainN.shape)


    eixoX_train3 = df_train3[timestamp]
    df_train3 = df_train3.drop(timestamp,axis=1)

    df_train = pd.concat([df_train, df_train3], ignore_index=True)
    eixoX_train = pd.concat([eixoX_train, eixoX_train3], ignore_index=True)

## Período 4

In [ ]:
#Data train 
start_date_train4 = pd.to_datetime('2025-03-29 14:40:00')
end_date_train4 = pd.to_datetime('2025-03-29 16:30:00')

mask = (df_dataset[timestamp] > start_date_train4) & (df_dataset[timestamp] <= end_date_train4)
df_train4 = df_dataset.loc[mask]

# pre - processamento
print(df_train4.shape)

list_periods_train = []
for pro in pre_process:
    df_train4,_,list_aux = ppd.drop_transitorio_desligado(df_train4,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train4.reset_index(inplace=True, drop=True)

print(df_train4.shape)


eixoX_train4 = df_train4[timestamp]
df_train4 = df_train4.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train4], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train4], ignore_index=True)

## Período 5

In [ ]:
#Data train 
start_date_train5 = pd.to_datetime('2025-04-07 18:00:00')
end_date_train5 = pd.to_datetime('2025-04-07 20:00:00')

mask = (df_dataset[timestamp] > start_date_train5) & (df_dataset[timestamp] <= end_date_train5)
df_train5 = df_dataset.loc[mask]

# pre - processamento
print(df_train5.shape)

list_periods_train = []
for pro in pre_process:
    df_train5,_,list_aux = ppd.drop_transitorio_desligado(df_train5,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train5.reset_index(inplace=True, drop=True)

print(df_train5.shape)


eixoX_train5 = df_train5[timestamp]
df_train5 = df_train5.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train5], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train5], ignore_index=True)

## Período 6

In [ ]:
#Data train 
start_date_train6 = pd.to_datetime('2025-05-18 03:50:00')
end_date_train6 = pd.to_datetime('2025-05-18 03:56:00')

for i in range(15):
    mask = (df_dataset[timestamp] > start_date_train6) & (df_dataset[timestamp] <= end_date_train6)
    df_train6 = df_dataset.loc[mask]

    # pre - processamento
    #print(df_trainN.shape)
    list_periods_train = []
    for pro in pre_process:
        df_train6,_,list_aux = ppd.drop_transitorio_desligado(df_train6,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
        list_periods_train = [*list_periods_train,*list_aux]

    df_train6.reset_index(inplace=True, drop=True)

    #print(df_trainN.shape)


    eixoX_train6 = df_train6[timestamp]
    df_train6 = df_train6.drop(timestamp,axis=1)

    df_train = pd.concat([df_train, df_train6], ignore_index=True)
    eixoX_train = pd.concat([eixoX_train, eixoX_train6], ignore_index=True)

## Período 7

In [ ]:
#Data train 
start_date_train7 = pd.to_datetime('2025-05-19 11:50:00')
end_date_train7 = pd.to_datetime('2025-05-19 12:10:00')

mask = (df_dataset[timestamp] > start_date_train7) & (df_dataset[timestamp] <= end_date_train7)
df_train7 = df_dataset.loc[mask]

# pre - processamento
print(df_train7.shape)

list_periods_train = []
for pro in pre_process:
    df_train7,_,list_aux = ppd.drop_transitorio_desligado(df_train7,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train7.reset_index(inplace=True, drop=True)

print(df_train7.shape)


eixoX_train7 = df_train7[timestamp]
df_train7 = df_train7.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train7], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train7], ignore_index=True)

## Período 8

In [ ]:
#Data train 
start_date_train8 = pd.to_datetime('2025-06-10 11:00:00')
end_date_train8 = pd.to_datetime('2025-06-10 16:10:00')

mask = (df_dataset[timestamp] > start_date_train8) & (df_dataset[timestamp] <= end_date_train8)
df_train8 = df_dataset.loc[mask]

# pre - processamento
print(df_train8.shape)

list_periods_train = []
for pro in pre_process:
    df_train8,_,list_aux = ppd.drop_transitorio_desligado(df_train8,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train8.reset_index(inplace=True, drop=True)

print(df_train8.shape)


eixoX_train8 = df_train8[timestamp]
df_train8 = df_train8.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train8], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train8], ignore_index=True)

## Período 9

In [ ]:
#Data train 
start_date_train9 = pd.to_datetime('2025-07-30 03:00:00')
end_date_train9 = pd.to_datetime('2025-07-30 03:30:00')

mask = (df_dataset[timestamp] > start_date_train9) & (df_dataset[timestamp] <= end_date_train9)
df_train9 = df_dataset.loc[mask]

# pre - processamento
print(df_train9.shape)

list_periods_train = []
for pro in pre_process:
    df_train9,_,list_aux = ppd.drop_transitorio_desligado(df_train9,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train9.reset_index(inplace=True, drop=True)

print(df_train9.shape)


eixoX_train9 = df_train9[timestamp]
df_train9 = df_train9.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train9], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train9], ignore_index=True)

## Período 10

In [ ]:
#Data train 
start_date_train10 = pd.to_datetime('2025-08-20 05:47:00')
end_date_train10 = pd.to_datetime('2025-08-20 05:55:00')

for i in range(15):
    mask = (df_dataset[timestamp] > start_date_train10) & (df_dataset[timestamp] <= end_date_train10)
    df_train10 = df_dataset.loc[mask]

    # pre - processamento
    #print(df_trainN.shape)
    list_periods_train = []
    for pro in pre_process:
        df_train10,_,list_aux = ppd.drop_transitorio_desligado(df_train10,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
        list_periods_train = [*list_periods_train,*list_aux]

    df_train10.reset_index(inplace=True, drop=True)

    #print(df_trainN.shape)


    eixoX_train10 = df_train10[timestamp]
    df_train10 = df_train10.drop(timestamp,axis=1)

    df_train = pd.concat([df_train, df_train10], ignore_index=True)
    eixoX_train = pd.concat([eixoX_train, eixoX_train10], ignore_index=True)

## Período 11

In [ ]:
#Data train 
start_date_train11 = pd.to_datetime('2025-08-25 14:55:00')
end_date_train11 = pd.to_datetime('2025-08-25 15:05:00')

for i in range(15):
    mask = (df_dataset[timestamp] > start_date_train11) & (df_dataset[timestamp] <= end_date_train11)
    df_train11 = df_dataset.loc[mask]

    # pre - processamento
    #print(df_trainN.shape)
    list_periods_train = []
    for pro in pre_process:
        df_train11,_,list_aux = ppd.drop_transitorio_desligado(df_train11,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
        list_periods_train = [*list_periods_train,*list_aux]

    df_train11.reset_index(inplace=True, drop=True)

    #print(df_trainN.shape)


    eixoX_train11 = df_train11[timestamp]
    df_train11 = df_train11.drop(timestamp,axis=1)

    df_train = pd.concat([df_train, df_train11], ignore_index=True)
    eixoX_train = pd.concat([eixoX_train, eixoX_train11], ignore_index=True)

## Período 12

In [ ]:
#Data train 
start_date_train12 = pd.to_datetime('2026-01-09 03:00:00')
end_date_train12 = pd.to_datetime('2026-01-09 05:00:00')

mask = (df_dataset[timestamp] > start_date_train12) & (df_dataset[timestamp] <= end_date_train12)
df_train12 = df_dataset.loc[mask]

# pre - processamento
print(df_train12.shape)

list_periods_train = []
for pro in pre_process:
    df_train12,_,list_aux = ppd.drop_transitorio_desligado(df_train12,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train12.reset_index(inplace=True, drop=True)

print(df_train12.shape)


eixoX_train12 = df_train12[timestamp]
df_train12 = df_train12.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train12], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train12], ignore_index=True)

## Período 13

In [ ]:
#Data train 
start_date_train13 = pd.to_datetime('2026-01-18 20:53:00')
end_date_train13 = pd.to_datetime('2026-01-18 20:59:00')

for i in range(10):
    mask = (df_dataset[timestamp] > start_date_train13) & (df_dataset[timestamp] <= end_date_train13)
    df_train13 = df_dataset.loc[mask]

    # pre - processamento
    #print(df_trainN.shape)
    list_periods_train = []
    for pro in pre_process:
        df_train13,_,list_aux = ppd.drop_transitorio_desligado(df_train13,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
        list_periods_train = [*list_periods_train,*list_aux]

    df_train13.reset_index(inplace=True, drop=True)

    #print(df_trainN.shape)


    eixoX_train13 = df_train13[timestamp]
    df_train13 = df_train13.drop(timestamp,axis=1)

    df_train = pd.concat([df_train, df_train13], ignore_index=True)
    eixoX_train = pd.concat([eixoX_train, eixoX_train13], ignore_index=True)

## Período 14

In [ ]:
#Data train 
start_date_train14 = pd.to_datetime('2026-01-13 18:55:00')
end_date_train14 = pd.to_datetime('2026-01-13 20:15:00')

for i in range(3):
    mask = (df_dataset[timestamp] > start_date_train14) & (df_dataset[timestamp] <= end_date_train14)
    df_train14 = df_dataset.loc[mask]

    # pre - processamento
    #print(df_trainN.shape)
    list_periods_train = []
    for pro in pre_process:
        df_train14,_,list_aux = ppd.drop_transitorio_desligado(df_train14,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
        list_periods_train = [*list_periods_train,*list_aux]

    df_train14.reset_index(inplace=True, drop=True)

    #print(df_trainN.shape)


    eixoX_train14 = df_train14[timestamp]
    df_train14 = df_train14.drop(timestamp,axis=1)

    df_train = pd.concat([df_train, df_train14], ignore_index=True)
    eixoX_train = pd.concat([eixoX_train, eixoX_train14], ignore_index=True)

# Data Test

In [ ]:
## Problema não detectado em Fev/2025
start_date = pd.to_datetime("2025-02-01 00:00:00")
end_date = pd.to_datetime("2025-04-01 00:00:00")

mask = (df_dataset[timestamp] > start_date) & (df_dataset[timestamp] <= end_date)
df_test = df_dataset.loc[mask]

list_periods_nan = ppd.select_periods_nan(df_dataset,timestamp)

# pre - processamento
print(df_test.shape)

list_periods_test = []
for pro in pre_process:
    df_test,_,list_aux = ppd.drop_transitorio_desligado(df_test,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_test = [*list_periods_test,*list_aux]
    
print(df_test.shape)

df_test_aux = df_test.copy()
eixoX_test = df_test[timestamp]
df_test = df_test.drop(timestamp,axis=1)


df_test.reset_index(inplace=True, drop=True)

# Anomaly Detection

## PCA $\phi$(T$^2$ + Q)

### Fit

In [ ]:
# Instacinamento da Classe
gain = 1.2
nc = 0
futurai = FuturaiML(nc,gain)
futurai.fit(df_train)

print("T²: {:.2f}".format(futurai.t2_lim))
print("Q: {:.2f}".format(futurai.q_lim))
print("Phi: {:.2f}".format(futurai.phi_lim))
print("Componentes: {:}".format(futurai.nc))

### Predict

In [ ]:
result = futurai.predict(df_test,eixoX_test)

fig_all_period, _ = utils.dev_graph_predict(result["phi"], result["timestamp"], futurai.phi_lim, " ", start_date, end_date, list_periods=None, plot_anomalies=False)
fig_all_period.show()

### Contribution

In [ ]:
## Aumento de vibração no GARO
# start_date_contribution = pd.to_datetime("2025-02-17 10:40:00")
# end_date_contribution = pd.to_datetime("2025-02-19 03:08:00")

## Aomalia do dia 09/03, gráfico indo para valores muito acima
start_date_contribution = pd.to_datetime("2025-03-09 08:45:00")
end_date_contribution = pd.to_datetime("2025-03-09 18:00:00")

mask = (df_dataset[timestamp] >= start_date_contribution) & (df_dataset[timestamp] <= end_date_contribution)
df_test_contribution = df_dataset.loc[mask]

# pre - processamento
print(df_test_contribution.shape)
list_periods_test = []
for pro in pre_process:
    df_test_contribution,_,list_aux = ppd.drop_transitorio_desligado(df_test_contribution,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_test = [*list_periods_test,*list_aux]
print(df_test_contribution.shape)

df_test_contribution_aux = df_test_contribution.copy()
eixoX_test_contribution = df_test_contribution[timestamp]
df_test_contribution = df_test_contribution.drop(timestamp,axis=1)

df_test_contribution.reset_index(inplace=True, drop=True)
result = futurai.predict(df_test_contribution,eixoX_test_contribution)

In [ ]:
dict_score_prin, dict_contribuicao, df_full, _ = futurai.contribuition(df_test_contribution,result["matrix"], df_sistema, result["timestamp"], eixoX_test_contribution)
pd.DataFrame().from_dict(dict_score_prin).head(10)

In [ ]:
# All period contribution
start_date_contribution = pd.to_datetime("2025-02-01 00:00:00")
end_date_contribution = pd.to_datetime("2025-04-01 00:00:00")

mask = (df_dataset[timestamp] >= start_date_contribution) & (df_dataset[timestamp] <= end_date_contribution)
df_test_contribution = df_dataset.loc[mask]

# pre - processamento
print(df_test_contribution.shape)
list_periods_test = []
for pro in pre_process:
    df_test_contribution,_,list_aux = ppd.drop_transitorio_desligado(df_test_contribution,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_test = [*list_periods_test,*list_aux]
print(df_test_contribution.shape)

df_test_contribution_aux = df_test_contribution.copy()
eixoX_test_contribution = df_test_contribution[timestamp]
df_test_contribution = df_test_contribution.drop(timestamp,axis=1)

df_test_contribution.reset_index(inplace=True, drop=True)
result = futurai.predict(df_test_contribution,eixoX_test_contribution)
_, _, _, df_projection_prin_vars = futurai.contribuition(df_test_contribution,result["matrix"], df_sistema, result["timestamp"], eixoX_test_contribution)

In [ ]:
mask = (df_dataset[timestamp] >= start_date_contribution) & (df_dataset[timestamp] <= end_date_contribution)
df_aux = df_dataset.loc[mask]
eixoX_aux = df_aux[timestamp]
df_aux = df_aux.drop(timestamp,axis=1)
df_aux.reset_index(inplace=True, drop=True)

lista = ["ARA-384V984.PV"]
fig = utils.graph_variables(df_aux, eixoX_aux,lista, pre_process=None, df_projection=df_projection_prin_vars)
fig.show()

### Plot Graph

In [ ]:
start_date = pd.to_datetime("2025-02-01 00:00:00")
end_date = pd.to_datetime("2025-03-01 00:00:00")
mask = (df_dataset[timestamp] > start_date) & (df_dataset[timestamp] <= end_date)
df_aux = df_dataset.loc[mask]
eixoX_aux = df_aux[timestamp]
df_aux = df_aux.drop(timestamp,axis=1)
df_aux.reset_index(inplace=True, drop=True)

lista = ["ARA-384V984.PV","ARA-384T927.PV", "ARA-384T671B.PV", "ARA-384I607.PV", "ARA-384P930.PV"]
fig = utils.graph_variables(df_aux, eixoX_aux,lista, pre_process=None)
fig.show()

## MSCRED
Chuxu Zhang, Dongjin Song, Yuncong Chen, Xinyang Feng, Cristian Lumezanu, Wei Cheng, Jingchao Ni, Bo Zong, Haifeng Chen, and Nitesh V. Chawla. 2019. A deep neural network for unsupervised anomaly detection and diagnosis in multivariate time series data. In Proceedings of the Thirty-Third AAAI Conference on Artificial Intelligence and Thirty-First Innovative Applications of Artificial Intelligence Conference and Ninth AAAI Symposium on Educational Advances in Artificial Intelligence (AAAI'19/IAAI'19/EAAI'19), Vol. 33. AAAI Press, Article 174, 1409–1416. https://doi.org/10.1609/aaai.v33i01.33011409

In [ ]:
model_config = {
    'batch_size': 128,
    'sensor_n': df_test.shape[1],
    'win_size': [10, 30, 60],
    'step_max': 5,
    'gap_time': 1,
    'learning_rate': 0.0003,
    'epochs': 5,
    'model_save_path': 'mscred_model/',
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

def get_scaler(df_list):
    full_df = pd.concat(df_list, axis=0)
    
    # Transform to (Sensores, Tempo)
    data = np.array(full_df.values.T, dtype=np.float64)
    
    # Global min/max on training set
    max_val = np.max(data, axis=1, keepdims=True)
    min_val = np.min(data, axis=1, keepdims=True)
    
    return min_val, max_val

def generate_signature_matrix(df_block, model_config, scaler_params):
    step_max = model_config['step_max']
    gap_time = model_config['gap_time']
    win_size = model_config['win_size']
    
    # Preprocess
    data = np.array(df_block.values.T, dtype=np.float64)
    sensor_n = data.shape[0]
    total_time = data.shape[1]
    
    # MinMax Normalization
    min_val, max_val = scaler_params
    epsilon = 1e-6
    data = (data - min_val) / (max_val - min_val + epsilon)

    # Generate Signature Matrix
    data_all_scales = []
    for w in range(len(win_size)):
        matrix_list = []
        win = win_size[w]
        
        for t in range(0, total_time, gap_time):
            matrix_t = np.zeros((sensor_n, sensor_n))
            
            # t < win, generate zeros (padding), however the loop ignore this initial zeros
            if t >= win:
                segment = data[:, t - win : t]
                matrix_t = np.matmul(segment, segment.T) / win
            
            matrix_list.append(matrix_t)
        data_all_scales.append(matrix_list)

    # ConvLSTM
    X_block = []
    total_generated_steps = len(data_all_scales[0])
    num_scales = len(win_size)
    
    # Valid sequences when MAIOR JANELA + SEQUÊNCIA HISTÓRICA
    start_idx = (win_size[-1] // gap_time) + step_max

    for i in range(total_generated_steps):
        # Jump cold start
        if i < start_idx:
            continue
            
        sequence_matrices = [] 
        for step in range(step_max, 0, -1):
            idx = i - step
            multi_scale_step = []
            for scale_idx in range(num_scales):
                multi_scale_step.append(data_all_scales[scale_idx][idx])
            sequence_matrices.append(multi_scale_step)
        
        X_block.append(np.array(sequence_matrices))

    if len(X_block) > 0:
        return np.array(X_block)
    else:
        return np.empty((0)) # Retorna vazio se o bloco for muito pequeno


df_train_list = [
    df_train1,
    df_train2,
    df_train3,
    df_train4,
    df_train5,
    df_train6,
    df_train7,
    df_train8,
    df_train9,
    df_train10,
    df_train11,
    df_train12,
    df_train13,
    df_train14
]

df_test_list = [
    df_test
]

# Get scaler values
scaler = get_scaler(df_train_list)

X_train_list = []
for i, block in enumerate(df_train_list):
    print(f"df_train{i+1} shape {len(block)}...")
    X_proc = generate_signature_matrix(block, model_config, scaler)
    
    if X_proc.size > 0:
        X_train_list.append(X_proc)
        print(f" -> {X_proc.shape[0]} sequences.")

if X_train_list:
    X_train_final = np.concatenate(X_train_list, axis=0)
    print(f"\nFinal shape training data: {X_train_final.shape}")

### Fit

In [ ]:
print(f"Using device: {model_config['device']}")
class MemoryDataset(Dataset):
    def __init__(self, data_array):
        """
        Args:
            data_array: Numpy array with shape (Samples, Steps, Scales, H, W)
        """
        self.data = data_array

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        # (Steps, Scales, H, W)
        sample = self.data[idx]
        
        # Convert to Tensor float32
        return torch.from_numpy(sample).float()

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = kernel_size[0] // 2, kernel_size[1] // 2
        self.bias = bias
        self.conv = nn.Conv2d(in_channels=self.input_dim + self.hidden_dim,
                              out_channels=4 * self.hidden_dim,
                              kernel_size=self.kernel_size,
                              padding=self.padding,
                              bias=self.bias)

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

    def init_hidden(self, batch_size, image_size, device):
        height, width = image_size
        return (torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
                torch.zeros(batch_size, self.hidden_dim, height, width, device=device))

class TemporalAttention(nn.Module):
    def __init__(self):
        super(TemporalAttention, self).__init__()

    def forward(self, hidden_states):
        # hidden_states: [batch, steps, channels, H, W]
        last_step = hidden_states[:, -1, ...]
        steps = hidden_states.shape[1]
        weights = []
        for k in range(steps):
            curr_step = hidden_states[:, k, ...]
            # Eq (4) Dot Product
            dot_product = torch.sum(curr_step * last_step, dim=(1, 2, 3)) 
            weights.append(dot_product / steps)
        
        weights = torch.stack(weights, dim=1)
        weights = F.softmax(weights, dim=1).unsqueeze(2).unsqueeze(3).unsqueeze(4)
        context = torch.sum(hidden_states * weights, dim=1)
        return context

class MSCRED(nn.Module):
    def __init__(self, sensor_n, scale_n, step_max):
        super(MSCRED, self).__init__()
        self.sensor_n = sensor_n
        
        # Encoder
        self.conv1 = nn.Conv2d(scale_n, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=2, stride=2, padding=0)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=2, stride=2, padding=0)
        
        # ConvLSTMs
        self.lstm1 = ConvLSTMCell(32, 32, (3,3), True)
        self.lstm2 = ConvLSTMCell(64, 64, (3,3), True)
        self.lstm3 = ConvLSTMCell(128, 128, (3,3), True)
        self.lstm4 = ConvLSTMCell(256, 256, (3,3), True)
        
        self.attention = TemporalAttention()
        
        # Decoder
        self.deconv4 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2, padding=0)
        self.deconv3 = nn.ConvTranspose2d(256, 64, kernel_size=2, stride=2, padding=0)
        self.deconv2 = nn.ConvTranspose2d(128, 32, kernel_size=3, stride=2, padding=1)
        self.deconv1 = nn.ConvTranspose2d(64, scale_n, kernel_size=3, stride=1, padding=1)
        
    def forward(self, x):
        b, steps, c, h, w = x.size()
        h1_list, h2_list, h3_list, h4_list = [], [], [], []
        
        # init hidden states in the first loop
        # Calculate expected shapes after convolutions for init_hidden
        dummy_in = torch.zeros(1, c, h, w, device=x.device)
        d1 = self.conv1(dummy_in)
        d2 = self.conv2(d1)
        d3 = self.conv3(d2)
        d4 = self.conv4(d3)
        
        state1 = self.lstm1.init_hidden(b, (d1.size(2), d1.size(3)), x.device)
        state2 = self.lstm2.init_hidden(b, (d2.size(2), d2.size(3)), x.device)
        state3 = self.lstm3.init_hidden(b, (d3.size(2), d3.size(3)), x.device)
        state4 = self.lstm4.init_hidden(b, (d4.size(2), d4.size(3)), x.device)

        for t in range(steps):
            input_t = x[:, t, :, :, :]
            enc1 = F.selu(self.conv1(input_t))
            enc2 = F.selu(self.conv2(enc1))
            enc3 = F.selu(self.conv3(enc2))
            enc4 = F.selu(self.conv4(enc3))
            
            h1, c1 = self.lstm1(enc1, state1)
            state1 = (h1, c1)
            h1_list.append(h1)
            
            h2, c2 = self.lstm2(enc2, state2)
            state2 = (h2, c2)
            h2_list.append(h2)
            
            h3, c3 = self.lstm3(enc3, state3)
            state3 = (h3, c3)
            h3_list.append(h3)
            
            h4, c4 = self.lstm4(enc4, state4)
            state4 = (h4, c4)
            h4_list.append(h4)
            
        h1_stack = torch.stack(h1_list, dim=1)
        h2_stack = torch.stack(h2_list, dim=1)
        h3_stack = torch.stack(h3_list, dim=1)
        h4_stack = torch.stack(h4_list, dim=1)
        
        attn1 = self.attention(h1_stack)
        attn2 = self.attention(h2_stack)
        attn3 = self.attention(h3_stack)
        attn4 = self.attention(h4_stack)
        
        dec4 = F.selu(self.deconv4(attn4))
        if dec4.shape[2:] != attn3.shape[2:]: dec4 = F.interpolate(dec4, size=attn3.shape[2:])
        dec4_concat = torch.cat([dec4, attn3], dim=1)
        
        dec3 = F.selu(self.deconv3(dec4_concat))
        if dec3.shape[2:] != attn2.shape[2:]: dec3 = F.interpolate(dec3, size=attn2.shape[2:])
        dec3_concat = torch.cat([dec3, attn2], dim=1)
        
        dec2 = F.selu(self.deconv2(dec3_concat))
        if dec2.shape[2:] != attn1.shape[2:]: dec2 = F.interpolate(dec2, size=attn1.shape[2:])
        dec2_concat = torch.cat([dec2, attn1], dim=1)
        
        output = F.selu(self.deconv1(dec2_concat))
        return output


def train_model(X_train, model_config):
    device = model_config['device']
    
    if X_train is None or len(X_train) == 0:
        print("X_train is empty!")
        return None

    # Dataset e DataLoader
    dataset = MemoryDataset(X_train)
    dataloader = DataLoader(dataset, batch_size=model_config['batch_size'], shuffle=True)
    
    # Automatically identify dimensions
    # Expected Shape: [Samples, Steps, Scales, H, W]
    sample_shape = X_train.shape
    scale_n = sample_shape[2]
    sensor_n = sample_shape[3] # H = W = sensors
    
    print(f"Init training process... \n(Scales: {scale_n}, Sensors: {sensor_n})")

    model_mscred = MSCRED(sensor_n=sensor_n, scale_n=scale_n, step_max=model_config['step_max']).to(device)
    optimizer = torch.optim.Adam(model_mscred.parameters(), lr=model_config['learning_rate'])
    criterion = nn.MSELoss()
    
    model_mscred.train()
    for epoch in range(model_config['epochs']):
        epoch_loss = 0
        start_time = time.time()
        
        for i, batch_x in enumerate(dataloader):
            batch_x = batch_x.to(device)
            target = batch_x[:, -1, :, :, :] # Target é reconstruir a última matriz
            
            optimizer.zero_grad()
            reconstruction = model_mscred(batch_x)
            
            loss = criterion(reconstruction, target)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{model_config['epochs']}] Loss: {avg_loss:.6f} Time: {time.time()-start_time:.2f}s")
        
    # Save final model
    # torch.save(model_mscred.state_dict(), "mscred_final.pth")
    return model_mscred

# Train model
model_mscred = train_model(X_train_final, model_config)

### Predict

In [ ]:
# Predict
def predict_mscred(df_test, timestamps, model, model_config, scaler_params):
    device = next(model.parameters()).device
    
    # Geração de Matrizes
    X_test = generate_signature_matrix(df_test, model_config, scaler_params)
    
    if len(X_test) == 0:
        print("X_test vazio. Nenhum dado gerado.")
        return pd.DataFrame()
        
    # DataLoader
    test_dataset = MemoryDataset(X_test)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)
    
    # Cálculo da Reconstrução
    model.eval()
    batch_scores = []
    with torch.no_grad():
        for i, batch_x in enumerate(test_loader):
            batch_x_gpu = batch_x.to(device)
            
            # Inferência
            recon_gpu = model(batch_x_gpu)
            
            # Cálculo do Erro (Sem guardar a matriz reconstruída)
            # Trazemos de volta para CPU apenas para cálculo do erro
            recon_np = recon_gpu.cpu().numpy()
            original_np = batch_x.numpy() # Já está na CPU pelo DataLoader
            
            # Ground Truth do batch atual (Último passo da sequência)
            # shape: (Batch, Steps, Scales, H, W) -> Pega step -1
            gt_batch = original_np[:, -1, :, :, :]
            
            # Calcula MSE do batch
            # Shape: (Batch, Scales, H, W)
            diff = np.square(gt_batch - recon_np)
            
            # Média sobre eixos (Scales, H, W) -> Resultado: vetor de tamanho (Batch,)
            scores = np.mean(diff, axis=(1, 2, 3))
            
            # Guarda os scores, não as matrizes
            batch_scores.append(scores)
            
            # Limpeza de memória
            del batch_x_gpu, recon_gpu, recon_np, gt_batch, diff
            
    scores_test = np.concatenate(batch_scores, axis=0)
    del X_test
    gc.collect()
    
    # Alinhamento Temporal
    gap_time = model_config['gap_time']
    win_size = model_config['win_size']
    step_max = model_config['step_max']
    
    # Calcular o offset inicial (dados descartados no início)
    start_gap_steps = (win_size[-1] // gap_time) + step_max
    start_index_real = start_gap_steps * gap_time
    
    # Preparar timestamps
    if hasattr(timestamps, 'values'):
        ts_values = timestamps.values
    else:
        ts_values = np.array(timestamps)
        
    # Fatiamento correto: começa no índice real e aplica o passo (gap_time)
    valid_timestamps = ts_values[start_index_real :: gap_time]
    
    # Corte de segurança para garantir tamanhos iguais
    min_len = min(len(scores_test), len(valid_timestamps))
    
    # Criação do DataFrame final
    df_predict = pd.DataFrame({
        'timestamp': valid_timestamps[:min_len],
        'score': scores_test[:min_len]
    })
    df_predict.set_index('timestamp', inplace=True)
    df_predict = df_predict.asfreq('1min')
    df_predict['score'] = df_predict['score'].fillna(0)
    
    return df_predict

df_predict_mscred = predict_mscred(
    df_test=df_test,
    timestamps=eixoX_test,
    model=model_mscred,
    model_config=model_config,
    scaler_params=scaler
)

In [ ]:
def get_train_scores(X_train, model, batch_size=32):
    """
    Calcula os scores de reconstrução do treino por batch para economizar memória.
    Necessário para definir o limiar sem travar o kernel.
    """
    device = next(model.parameters()).device
    
    # Otimização de tipo
    if X_train.dtype != np.float32:
        X_test = X_train.astype(np.float32)
        
    dataset = MemoryDataset(X_train)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    model.eval()
    all_scores = []
    
    with torch.no_grad():
        for batch_x in loader:
            # Envia para GPU
            batch_x_gpu = batch_x.to(device)
            
            # Reconstrói
            recon_gpu = model(batch_x_gpu)
            
            # Traz para CPU para cálculo
            recon_np = recon_gpu.cpu().numpy()
            original_np = batch_x.numpy()
            
            # Pega o Ground Truth (último passo)
            gt_batch = original_np[:, -1, :, :, :]
            
            # Calcula MSE deste batch
            diff = np.square(gt_batch - recon_np)
            
            # Média (Score)
            scores = np.mean(diff, axis=(1, 2, 3))
            
            all_scores.append(scores)
            
            # Limpa memória GPU
            del batch_x_gpu, recon_gpu
            
    return np.concatenate(all_scores, axis=0)

train_scores = get_train_scores(X_train_final, model_mscred, batch_size=128)
percentile = 99 
threshold = np.percentile(train_scores, percentile)

print(f"Média do Score (Treino): {np.mean(train_scores):.6f}")
print(f"Máximo do Score (Treino): {np.max(train_scores):.6f}")
print(f"Limiar Definido (P{percentile}): {threshold:.6f}")

In [ ]:
def plot_predict_mscred(df_predict, threshold):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_predict.index,
        y=df_predict['score'],
        mode='lines',
        name='Erro de Reconstrução',
        fill="tozeroy", 
        line=dict(color="#0F293A"),
        connectgaps=False
    ))
    
    fig.add_hline(
        y=threshold, 
        name='Limiar',
        line_dash="dash", 
        line_color="red", 
        line_width=2
    )
    
    fig.update_layout(
        xaxis_title='Data/Hora',
        yaxis_title='Erro Médio Absoluto (MAE)',
        template='plotly_white',
        hovermode="x unified",
        showlegend=True
    )

    return fig

fig = plot_predict_mscred(df_predict_mscred, threshold)
fig.show()

### Contribution

In [ ]:
def contribution_mscred(model, df_test, timestamps, config, scaler_params, start_date, end_date):
    device = next(model.parameters()).device
    model.eval()
    
    if not isinstance(timestamps, pd.Index):
        timestamps = pd.to_datetime(timestamps)
        
    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)
    
    mask_period = (timestamps >= start_dt) & (timestamps <= end_dt)
    if not mask_period.any():
        raise ValueError(f"Nenhum dado encontrado entre {start_date} e {end_date}")
        
    valid_indices = np.where(mask_period)[0]
    idx_start = valid_indices[0]
    idx_end = valid_indices[-1]
    
    # Buffer (Histórico necessário para o MSCRED funcionar)
    gap_time = config['gap_time']
    win_size = config['win_size']
    step_max = config['step_max']
    
    # Buffer: Maior Janela + Sequência LSTM + Margem de segurança
    buffer_steps = (win_size[-1] // gap_time) + step_max + 2
    buffer_indices = buffer_steps * gap_time
    
    # Define o slice com buffer
    slice_start = max(0, idx_start - buffer_indices)
    slice_end = idx_end + 1
    
    df_subset = df_test.iloc[slice_start : slice_end]
    
    # Geração das Matrizes para o Período
    X_subset = generate_signature_matrix(df_subset, config, scaler_params)
    
    if len(X_subset) == 0:
        raise ValueError("Intervalo muito curto para gerar matrizes.")

    # Inferência e Acumulação de Erro
    dataset = MemoryDataset(X_subset)
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    
    # Matriz de erro: (Scales, Sensores, Sensores)
    total_error_matrix = np.zeros((config['sensor_n'], config['sensor_n']), dtype=np.float64)
    
    samples_count = 0
    with torch.no_grad():
        for batch_x in loader:
            batch_x = batch_x.to(device)
            
            # Reconstrução da matrix
            recon = model(batch_x)
            
            recon_np = recon.cpu().numpy()
            input_np = batch_x.cpu().numpy()
            
            # Ground Truth (Último passo)
            gt_batch = input_np[:, -1, :, :, :]
            
            # Erro Quadrático (Batch, Scales, H, W)
            diff = np.square(gt_batch - recon_np)
            
            # Média sobre as Escalas (Dimensão 1) -> (Batch, H, W)
            diff_aggregated_scales = np.mean(diff, axis=1)
            
            # Soma sobre o Tempo (Batch) -> (H, W)
            batch_total_error = np.sum(diff_aggregated_scales, axis=0)
            
            total_error_matrix += batch_total_error
            samples_count += batch_x.shape[0]

    # Cálculo da Contribuição por Sensor
    # A matriz total_error_matrix é (Sensores, Sensores).
    # A linha i representa a soma dos erros de correlação do sensor i com todos os outros.
    # Soma a linha (axis=1) para ter um score único por sensor.
    sensor_scores = np.sum(total_error_matrix, axis=1)
    total_period_error = np.sum(sensor_scores)
    
    variable_names = df_test.columns
    diagnosis_df = pd.DataFrame({
        'Variable': variable_names,
        'Total_Error_Contribution': sensor_scores,
        'Contribution %': (sensor_scores / total_period_error) * 100
    })
    diagnosis_df = diagnosis_df.sort_values(by='Total_Error_Contribution', ascending=False).reset_index(drop=True)
    
    return diagnosis_df

In [ ]:
start_anomaly = '2025-02-17 15:30:00'
end_anomaly = '2025-02-19 03:08:00'

df_contribution_mscred = contribution_mscred(
    model=model_mscred,
    df_test=df_test,
    timestamps=eixoX_test,
    config=model_config,
    scaler_params=scaler,
    start_date=start_anomaly,
    end_date=end_anomaly
)
df_contribution_mscred

## MSCRED [Adaptação híbrida para recuperar reconstrução das variáveis]
Chuxu Zhang, Dongjin Song, Yuncong Chen, Xinyang Feng, Cristian Lumezanu, Wei Cheng, Jingchao Ni, Bo Zong, Haifeng Chen, and Nitesh V. Chawla. 2019. A deep neural network for unsupervised anomaly detection and diagnosis in multivariate time series data. In Proceedings of the Thirty-Third AAAI Conference on Artificial Intelligence and Thirty-First Innovative Applications of Artificial Intelligence Conference and Ninth AAAI Symposium on Educational Advances in Artificial Intelligence (AAAI'19/IAAI'19/EAAI'19), Vol. 33. AAAI Press, Article 174, 1409–1416. https://doi.org/10.1609/aaai.v33i01.33011409

In [ ]:
model_config = {
    'batch_size': 128,
    'sensor_n': df_test.shape[1],
    'win_size': [10, 30, 60],
    'step_max': 5,
    'gap_time': 1,
    'learning_rate': 0.0003,
    'epochs': 5,
    'model_save_path': 'mscred_model/',
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

def get_scaler(df_list):
    full_df = pd.concat(df_list, axis=0)
    
    # Transform to (Sensores, Tempo)
    data = np.array(full_df.values.T, dtype=np.float64)
    
    # Global min/max on training set
    max_val = np.max(data, axis=1, keepdims=True)
    min_val = np.min(data, axis=1, keepdims=True)
    
    return min_val, max_val

def generate_signature_matrix(df_block, model_config, scaler_params):
    step_max = model_config['step_max']
    gap_time = model_config['gap_time']
    win_size = model_config['win_size']
    
    # Preprocess
    data = np.array(df_block.values.T, dtype=np.float64)
    sensor_n = data.shape[0]
    total_time = data.shape[1]
    
    # MinMax Normalization
    min_val, max_val = scaler_params
    epsilon = 1e-6
    data = (data - min_val) / (max_val - min_val + epsilon)

    # Generate Signature Matrix
    data_all_scales = []
    for w in range(len(win_size)):
        matrix_list = []
        win = win_size[w]
        
        for t in range(0, total_time, gap_time):
            matrix_t = np.zeros((sensor_n, sensor_n))
            
            # t < win, generate zeros (padding), however the loop ignore this initial zeros
            if t >= win:
                segment = data[:, t - win : t]
                matrix_t = np.matmul(segment, segment.T) / win
            
            matrix_list.append(matrix_t)
        data_all_scales.append(matrix_list)

    # ConvLSTM
    X_block = []
    total_generated_steps = len(data_all_scales[0])
    num_scales = len(win_size)
    
    # Valid sequences when MAIOR JANELA + SEQUÊNCIA HISTÓRICA
    start_idx = (win_size[-1] // gap_time) + step_max

    for i in range(total_generated_steps):
        # Jump cold start
        if i < start_idx:
            continue
            
        sequence_matrices = [] 
        for step in range(step_max, 0, -1):
            idx = i - step
            multi_scale_step = []
            for scale_idx in range(num_scales):
                multi_scale_step.append(data_all_scales[scale_idx][idx])
            sequence_matrices.append(multi_scale_step)
        
        X_block.append(np.array(sequence_matrices))

    if len(X_block) > 0:
        return np.array(X_block)
    else:
        return np.empty((0)) # Retorna vazio se o bloco for muito pequeno


df_train_list = [
    df_train1,
    df_train2,
    df_train3,
    df_train4,
    df_train5,
    df_train6,
    df_train7,
    df_train8,
    df_train9,
    df_train10,
    df_train11,
    df_train12,
    df_train13,
    df_train14
]

df_test_list = [
    df_test
]

# Get scaler values
scaler = get_scaler(df_train_list)

X_train_list = []
for i, block in enumerate(df_train_list):
    print(f"df_train{i+1} shape {len(block)}...")
    X_proc = generate_signature_matrix(block, model_config, scaler)
    
    if X_proc.size > 0:
        X_train_list.append(X_proc)
        print(f" -> {X_proc.shape[0]} sequences.")

if X_train_list:
    X_train_final = np.concatenate(X_train_list, axis=0)
    print(f"\nFinal shape training data: {X_train_final.shape}")

### Fit

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = kernel_size[0] // 2, kernel_size[1] // 2
        self.bias = bias
        self.conv = nn.Conv2d(in_channels=self.input_dim + self.hidden_dim,
                              out_channels=4 * self.hidden_dim,
                              kernel_size=self.kernel_size,
                              padding=self.padding,
                              bias=self.bias)

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

    def init_hidden(self, batch_size, image_size, device):
        height, width = image_size
        return (torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
                torch.zeros(batch_size, self.hidden_dim, height, width, device=device))

class TemporalAttention(nn.Module):
    def __init__(self):
        super(TemporalAttention, self).__init__()

    def forward(self, hidden_states):
        # hidden_states: [batch, steps, channels, H, W]
        last_step = hidden_states[:, -1, ...]
        steps = hidden_states.shape[1]
        weights = []
        for k in range(steps):
            curr_step = hidden_states[:, k, ...]
            dot_product = torch.sum(curr_step * last_step, dim=(1, 2, 3)) 
            weights.append(dot_product / steps)
        
        weights = torch.stack(weights, dim=1)
        weights = F.softmax(weights, dim=1).unsqueeze(2).unsqueeze(3).unsqueeze(4)
        context = torch.sum(hidden_states * weights, dim=1)
        return context

class MSCRED_Hybrid(nn.Module):
    def __init__(self, sensor_n, scale_n, step_max):
        super(MSCRED_Hybrid, self).__init__()
        self.sensor_n = sensor_n
        
        # --- ENCODER ---
        self.conv1 = nn.Conv2d(scale_n, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=2, stride=2, padding=0)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=2, stride=2, padding=0)
        
        # --- CONVLSTM ---
        self.lstm1 = ConvLSTMCell(32, 32, (3,3), True)
        self.lstm2 = ConvLSTMCell(64, 64, (3,3), True)
        self.lstm3 = ConvLSTMCell(128, 128, (3,3), True)
        self.lstm4 = ConvLSTMCell(256, 256, (3,3), True)
        
        self.attention = TemporalAttention()
        
        # --- DECODER 1: MATRIZES ---
        self.deconv4 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2, padding=0)
        self.deconv3 = nn.ConvTranspose2d(256, 64, kernel_size=2, stride=2, padding=0)
        self.deconv2 = nn.ConvTranspose2d(128, 32, kernel_size=3, stride=2, padding=1)
        self.deconv1 = nn.ConvTranspose2d(64, scale_n, kernel_size=3, stride=1, padding=1)
        
        # --- DECODER 2: VALORES ---
        # Global Pooling reduz (Batch, 256, H, W) para (Batch, 256, 1, 1)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.value_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, sensor_n)
        )
        
    def forward(self, x):
        b, steps, c, h, w = x.size()
    
        dummy_in = torch.zeros(1, c, h, w, device=x.device)
        d1 = self.conv1(dummy_in); d2 = self.conv2(d1); d3 = self.conv3(d2); d4 = self.conv4(d3)
        
        state1 = self.lstm1.init_hidden(b, (d1.size(2), d1.size(3)), x.device)
        state2 = self.lstm2.init_hidden(b, (d2.size(2), d2.size(3)), x.device)
        state3 = self.lstm3.init_hidden(b, (d3.size(2), d3.size(3)), x.device)
        state4 = self.lstm4.init_hidden(b, (d4.size(2), d4.size(3)), x.device)

        h1_list, h2_list, h3_list, h4_list = [], [], [], []

        # Passagem Temporal (Encoder + LSTM)
        for t in range(steps):
            input_t = x[:, t, :, :, :]
            enc1 = F.selu(self.conv1(input_t))
            enc2 = F.selu(self.conv2(enc1))
            enc3 = F.selu(self.conv3(enc2))
            enc4 = F.selu(self.conv4(enc3))
            
            h1, c1 = self.lstm1(enc1, state1); state1 = (h1, c1); h1_list.append(h1)
            h2, c2 = self.lstm2(enc2, state2); state2 = (h2, c2); h2_list.append(h2)
            h3, c3 = self.lstm3(enc3, state3); state3 = (h3, c3); h3_list.append(h3)
            h4, c4 = self.lstm4(enc4, state4); state4 = (h4, c4); h4_list.append(h4)
            
        h1_stack = torch.stack(h1_list, dim=1); attn1 = self.attention(h1_stack)
        h2_stack = torch.stack(h2_list, dim=1); attn2 = self.attention(h2_stack)
        h3_stack = torch.stack(h3_list, dim=1); attn3 = self.attention(h3_stack)
        h4_stack = torch.stack(h4_list, dim=1); attn4 = self.attention(h4_stack)
        
        # --- Reconstrução de Matriz ---
        dec4 = F.selu(self.deconv4(attn4))
        if dec4.shape[2:] != attn3.shape[2:]: dec4 = F.interpolate(dec4, size=attn3.shape[2:])
        dec4_concat = torch.cat([dec4, attn3], dim=1)
        
        dec3 = F.selu(self.deconv3(dec4_concat))
        if dec3.shape[2:] != attn2.shape[2:]: dec3 = F.interpolate(dec3, size=attn2.shape[2:])
        dec3_concat = torch.cat([dec3, attn2], dim=1)
        
        dec2 = F.selu(self.deconv2(dec3_concat))
        if dec2.shape[2:] != attn1.shape[2:]: dec2 = F.interpolate(dec2, size=attn1.shape[2:])
        dec2_concat = torch.cat([dec2, attn1], dim=1)
        
        matrix_output = F.selu(self.deconv1(dec2_concat))
        
        # --- Reconstrução de Valor ---
        # attn4 contém o contexto temporal mais profundo e rico
        latent_features = self.global_pool(attn4)
        value_output = self.value_head(latent_features)
        
        return matrix_output, value_output

class HybridDataset(Dataset):
    def __init__(self, matrix_array, value_array):
        """
        Args:
            matrix_array: (Samples, Steps, Scales, H, W)
            value_array: (Samples, Sensors) - Valores alvo para regressão
        """
        self.matrices = matrix_array
        self.values = value_array

    def __len__(self):
        return self.matrices.shape[0]

    def __getitem__(self, idx):
        # Retorna tupla: (Input Matriz, Target Valor)
        return (torch.from_numpy(self.matrices[idx]).float(), 
                torch.from_numpy(self.values[idx]).float())

def prepare_hybrid_data(X_matrices, df_source, config):
    """
    Alinha as matrizes 5D geradas com os valores escalados originais do DataFrame.
    """
    gap_time = config['gap_time']
    win_size = config['win_size']
    step_max = config['step_max']
    
    # Calcula o índice onde começam os dados válidos no df original
    start_gap_steps = (win_size[-1] // gap_time) + step_max
    start_index_real = start_gap_steps * gap_time
    
    # Pegar os valores do DataFrame correspondentes a cada matriz gerada
    values_aligned = df_source.iloc[start_index_real :: gap_time].values
    
    min_len = min(len(X_matrices), len(values_aligned))
    
    X_final = X_matrices[:min_len]
    y_final = values_aligned[:min_len]
    
    print(f"Matrix Shape: {X_final.shape}, Values Shape: {y_final.shape}")
    return X_final, y_final


def train_hybrid_model(X_train, y_train, model_config):
    device = model_config['device']
    
    # Dataset Híbrido
    dataset = HybridDataset(X_train, y_train)
    dataloader = DataLoader(dataset, batch_size=model_config['batch_size'], shuffle=True)
    
    # Inicialização
    sample_shape = X_train.shape
    scale_n = sample_shape[2]
    sensor_n = sample_shape[3]
    
    model = MSCRED_Hybrid(sensor_n=sensor_n, scale_n=scale_n, step_max=model_config['step_max']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=model_config['learning_rate'])
    
    criterion = nn.MSELoss()
    model.train()
    
    for epoch in range(model_config['epochs']):
        epoch_loss = 0
        epoch_loss_m = 0
        epoch_loss_v = 0
        start_time = time.time()
        
        for batch_matrix, batch_values in dataloader:
            batch_matrix = batch_matrix.to(device)
            batch_values = batch_values.to(device) # Target valores (B, Sensors)
            
            # Ground Truth da Matriz (último passo da entrada)
            target_matrix = batch_matrix[:, -1, :, :, :]
            
            optimizer.zero_grad()
            
            # Forward (retorna 2 saídas)
            recon_matrix, recon_values = model(batch_matrix)
            
            # Loss 1: Reconstrução da Matriz
            loss_matrix = criterion(recon_matrix, target_matrix)
            
            # Loss 2: Reconstrução dos Valores
            loss_value = criterion(recon_values, batch_values)
            
            # Loss Total Ponderada
            # Peso alpha=0.5 para valores para não dominar a tarefa principal
            alpha = 0.5 
            total_loss = loss_matrix + (alpha * loss_value)
            
            total_loss.backward()
            optimizer.step()
            
            epoch_loss += total_loss.item()
            epoch_loss_m += loss_matrix.item()
            epoch_loss_v += loss_value.item()
            
        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{model_config['epochs']}] "
              f"Total Loss: {avg_loss:.6f} (Matriz: {epoch_loss_m/len(dataloader):.6f} | Valor: {epoch_loss_v/len(dataloader):.6f}) "
              f"Time: {time.time()-start_time:.2f}s")
        
    return model

def get_train_scores_hybrid(X_train, y_train, model, batch_size=32):
    # float32 para economia de memória
    if X_train.dtype != np.float32:
        X_data = X_train.astype(np.float32)
    else:
        X_data = X_train
        
    if y_train.dtype != np.float32:
        y_data = y_train.astype(np.float32)
    else:
        y_data = y_train
        
    dataset = HybridDataset(X_data, y_data)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    model.eval()
    all_scores = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            # tuple (Matriz, Valor)
            recon_matrix, _ = model(batch_x) 
        
            recon_np = recon_matrix.numpy()
            original_np = batch_x.numpy()
            
            # Cálculo do Erro (Matrizes)
            # shape: (Batch, Steps, Scales, H, W) -> Pega o última passo
            gt_batch = original_np[:, -1, :, :, :]
            
            diff = np.square(gt_batch - recon_np)
            
            # Média sobre eixos (Scales, H, W) -> (Batch,)
            scores = np.mean(diff, axis=(1, 2, 3))
            
            all_scores.append(scores)

            del recon_matrix
            
    return np.concatenate(all_scores, axis=0)

In [ ]:
df_train_full = pd.concat(df_train_list) 
df_train_values = df_train_full.values
min_val, max_val = scaler
df_train_norm = (df_train_values - min_val.T) / (max_val.T - min_val.T + 1e-6)
df_train_norm = pd.DataFrame(df_train_norm, columns=df_train_full.columns)

X_train_hybrid, y_train_hybrid = prepare_hybrid_data(X_train_final, df_train_norm, model_config)

model_hybrid = train_hybrid_model(X_train_hybrid, y_train_hybrid, model_config)

In [ ]:
class SPOT:
    def __init__(self, q=1e-4):
        self.proba = q
        self.extreme_quantile = None
        self.data = None
        self.init_data = None
        self.init_threshold = None
        self.peaks = None
        self.n = 0
        self.Nt = 0

    def fit(self, init_data, data):
        if isinstance(data, list): self.data = np.array(data)
        elif isinstance(data, np.ndarray): self.data = data
        elif isinstance(data, pd.Series): self.data = data.values
        else: return
        if isinstance(init_data, list): self.init_data = np.array(init_data)
        elif isinstance(init_data, np.ndarray): self.init_data = init_data
        elif isinstance(init_data, pd.Series): self.init_data = init_data.values
        elif isinstance(init_data, int):
            self.init_data = self.data[:init_data]
            self.data = self.data[init_data:]
        elif isinstance(init_data, float) and (init_data < 1) and (init_data > 0):
            r = int(init_data * data.size)
            self.init_data = self.data[:r]
            self.data = self.data[r:]
        else: return

    def add(self, data):
        if isinstance(data, list): data = np.array(data)
        elif isinstance(data, np.ndarray): data = data
        elif isinstance(data, pd.Series): data = data.values
        else: return
        self.data = np.append(self.data, data)

    def initialize(self, level=0.98, min_extrema=False, verbose=True):
        if min_extrema:
            self.init_data = -self.init_data
            self.data = -self.data
            level = 1 - level
        level = level - math.floor(level)
        n_init = self.init_data.size
        S = np.sort(self.init_data)
        self.init_threshold = S[int(level * n_init)]
        self.peaks = self.init_data[self.init_data > self.init_threshold] - self.init_threshold
        self.Nt = self.peaks.size
        self.n = n_init
        if verbose:
            print('Initial threshold : %s' % self.init_threshold)
            print('Number of peaks : %s' % self.Nt)
            print('Grimshaw maximum log-likelihood estimation ... ', end='')
        g, s, l = self._grimshaw()
        self.extreme_quantile = self._quantile(g, s)
        if verbose:
            print('Extreme quantile (probability = %s): %s' % (self.proba, self.extreme_quantile))

    def _rootsFinder(fun, jac, bounds, npoints, method):
        from scipy.optimize import minimize
        if method == 'regular':
            step = (bounds[1] - bounds[0]) / (npoints + 1)
            if step == 0: bounds, step = (0, 1e-4), 1e-5
            X0 = np.arange(bounds[0] + step, bounds[1], step)
        elif method == 'random':
            X0 = np.random.uniform(bounds[0], bounds[1], npoints)
        def objFun(X, f, jac):
            g = 0
            j = np.zeros(X.shape)
            i = 0
            for x in X:
                fx = f(x)
                g = g + fx ** 2
                j[i] = 2 * fx * jac(x)
                i = i + 1
            return g, j
        opt = minimize(lambda X: objFun(X, fun, jac), X0,
                       method='L-BFGS-B',
                       jac=True, bounds=[bounds] * len(X0))
        X = opt.x
        np.round(X, decimals=5)
        return np.unique(X)

    def _log_likelihood(Y, gamma, sigma):
        n = Y.size
        if gamma != 0:
            tau = gamma / sigma
            L = -n * math.log(sigma) - (1 + (1 / gamma)) * (np.log(1 + tau * Y)).sum()
        else:
            L = n * (1 + math.log(Y.mean()))
        return L

    def _grimshaw(self, epsilon=1e-8, n_points=10):
        def u(s): return 1 + np.log(s).mean()
        def v(s): return np.mean(1 / s)
        def w(Y, t):
            s = 1 + t * Y
            us = u(s); vs = v(s)
            return us * vs - 1
        def jac_w(Y, t):
            s = 1 + t * Y
            us = u(s); vs = v(s)
            jac_us = (1 / t) * (1 - vs)
            jac_vs = (1 / t) * (-vs + np.mean(1 / s ** 2))
            return us * jac_vs + vs * jac_us
        Ym = self.peaks.min()
        YM = self.peaks.max()
        Ymean = self.peaks.mean()
        a = -1 / YM
        if abs(a) < 2 * epsilon: epsilon = abs(a) / n_points
        a = a + epsilon
        b = 2 * (Ymean - Ym) / (Ymean * Ym)
        c = 2 * (Ymean - Ym) / (Ym ** 2)
        left_zeros = SPOT._rootsFinder(lambda t: w(self.peaks, t),
                                       lambda t: jac_w(self.peaks, t),
                                       (a + epsilon, -epsilon),
                                       n_points, 'regular')
        right_zeros = SPOT._rootsFinder(lambda t: w(self.peaks, t),
                                        lambda t: jac_w(self.peaks, t),
                                        (b, c),
                                        n_points, 'regular')
        zeros = np.concatenate((left_zeros, right_zeros))
        gamma_best = 0
        sigma_best = Ymean
        ll_best = SPOT._log_likelihood(self.peaks, gamma_best, sigma_best)
        for z in zeros:
            gamma = u(1 + z * self.peaks) - 1
            sigma = gamma / z
            ll = SPOT._log_likelihood(self.peaks, gamma, sigma)
            if ll > ll_best:
                gamma_best = gamma
                sigma_best = sigma
                ll_best = ll
        return gamma_best, sigma_best, ll_best

    def _quantile(self, gamma, sigma):
        r = self.n * self.proba / self.Nt
        if gamma != 0: return self.init_threshold + (sigma / gamma) * (pow(r, -gamma) - 1)
        else: return self.init_threshold - sigma * math.log(r)

    def run(self, with_alarm=True, dynamic=True):
        if self.n > self.init_data.size:
            print('Warning : the algorithm seems to have already been run, you should initialize before running again')
            return {}
        th = []
        alarm = []
        for i in range(self.data.size):
            if not dynamic:
                if self.data[i] > self.init_threshold and with_alarm:
                    self.extreme_quantile = self.init_threshold
                    alarm.append(i)
            else:
                if self.data[i] > self.extreme_quantile:
                    if with_alarm: alarm.append(i)
                    else:
                        self.peaks = np.append(self.peaks, self.data[i] - self.init_threshold)
                        self.Nt += 1
                        self.n += 1
                        g, s, l = self._grimshaw()
                        self.extreme_quantile = self._quantile(g, s)
                elif self.data[i] > self.init_threshold:
                    self.peaks = np.append(self.peaks, self.data[i] - self.init_threshold)
                    self.Nt += 1
                    self.n += 1
                    g, s, l = self._grimshaw()
                    self.extreme_quantile = self._quantile(g, s)
                else:
                    self.n += 1
            th.append(self.extreme_quantile)
        return {'thresholds': th, 'alarms': alarm}

def pot_eval(init_score, q=1e-4, level=0.02):
    """
    Calcula o limiar POT estático baseado apenas nos scores de treino.
    """
    lms = level
    while True:
        try:
            s = SPOT(q)
            s.fit(init_score, init_score)
            # Initialize realiza o cálculo pesado (Grimshaw) e define o limiar
            s.initialize(level=lms, min_extrema=False, verbose=False)
        except Exception:
            # Se o Grimshaw falhar em convergir,o nível é reduzido
            lms = lms * 0.999
        else:
            break
    return s.extreme_quantile

# Obter Scores de Treino (Calibração do POT)
train_scores = get_train_scores_hybrid(X_train_final, y_train_hybrid, model_hybrid, batch_size=32)
threshold = pot_eval(train_scores, q=1e-4, level=0.02)

# Aplicação do Ganho
# Ganho é um multiplicador empírico para reduzir falsos positivos
gain = 0.9
final_threshold = threshold * gain

print(f"\n--- Resultados POT ---")
print(f"Limiar Base (POT): {threshold:.6f}")
print(f"Ganho Aplicado: {gain}")
print(f"Limiar Final: {final_threshold:.6f}")

### Predict

In [ ]:
# Predict
def predict_mscred_hybrid(df_test, timestamps, model, model_config, scaler_params):    
    device = next(model.parameters()).device
    
    # Geração de Matrizes (X_test)
    X_test = generate_signature_matrix(df_test, model_config, scaler_params)
    
    if len(X_test) == 0:
        print("X_test vazio.")
        return pd.DataFrame()
    
    # float32 para economia de memória
    X_test = X_test.astype(np.float32)
    
    # Preparação dos Valores (y_test) para a classe HybridDataset
    # Normalização
    min_val, max_val = scaler_params
    # Transforma para numpy array float64 para precisão
    raw_values = np.array(df_test.values, dtype=np.float64)
    epsilon = 1e-6
    # MinMaxScaler
    values_norm = (raw_values.T - min_val) / (max_val - min_val + epsilon)
    values_norm = values_norm.T
    
    # Alinhamento Temporal (Corte inicial + Gap)
    gap_time = model_config['gap_time']
    win_size = model_config['win_size']
    step_max = model_config['step_max']
    
    # Calcula onde começam os dados válidos
    start_gap_steps = (win_size[-1] // gap_time) + step_max
    start_index_real = start_gap_steps * gap_time
    
    y_test = values_norm[start_index_real :: gap_time]
    min_len = min(len(X_test), len(y_test))
    X_test = X_test[:min_len]
    y_test = y_test[:min_len]
    y_test = y_test.astype(np.float32)

    # DataLoader
    test_dataset = HybridDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=model_config.get('batch_size', 128), shuffle=False)
    
    model.eval()
    matrix_scores = []
    
    with torch.no_grad():
        for i, (batch_x, batch_y) in enumerate(test_loader):
            batch_x_gpu = batch_x.to(device)
            
            # tuple (Recon Matriz, Recon Valor)
            recon_matrix_gpu, recon_values_gpu = model(batch_x_gpu)
            
            # Score de Anomalia (Baseado nas Matrizes)
            recon_matrix_np = recon_matrix_gpu.cpu().numpy()
            original_np = batch_x.numpy()
            
            # Ground Truth (Último passo da sequência)
            gt_matrix = original_np[:, -1, :, :, :]
            
            # MSE das Matrizes
            diff_matrix = np.square(gt_matrix - recon_matrix_np)
            scores_m = np.mean(diff_matrix, axis=(1, 2, 3)) # (Batch,)
            matrix_scores.append(scores_m)
            
            del batch_x_gpu, recon_matrix_gpu, recon_values_gpu, recon_matrix_np, diff_matrix
            
    final_scores = np.concatenate(matrix_scores, axis=0)
    
    del X_test, y_test
    gc.collect()
    
    if hasattr(timestamps, 'values'):
        ts_values = timestamps.values
    else:
        ts_values = np.array(timestamps)
        
    valid_timestamps = ts_values[start_index_real :: gap_time]
    
    min_len_final = min(len(final_scores), len(valid_timestamps))
    
    df_predict = pd.DataFrame({
        'timestamp': valid_timestamps[:min_len_final],
        'score': final_scores[:min_len_final]
    })
    df_predict.set_index('timestamp', inplace=True)
    df_predict = df_predict.asfreq('1min')
    df_predict['score'] = df_predict['score'].fillna(0)

    return df_predict

df_predict_mscred_hybrid = predict_mscred_hybrid(
    df_test=df_test,
    timestamps=eixoX_test,
    model=model_hybrid,
    model_config=model_config,
    scaler_params=scaler
)

In [ ]:
def plot_predict_mscred(df_predict, threshold):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_predict.index,
        y=df_predict['score'],
        mode='lines',
        name='Erro de Reconstrução',
        fill="tozeroy", 
        line=dict(color="#0F293A"),
        connectgaps=False
    ))
    
    fig.add_hline(
        y=threshold, 
        name='Limiar',
        line_dash="dash", 
        line_color="red", 
        line_width=2
    )
    
    fig.update_layout(
        xaxis_title='Data/Hora',
        yaxis_title='Erro Médio Absoluto (MAE)',
        template='plotly_white',
        hovermode="x unified",
        showlegend=True
    )

    return fig

fig = plot_predict_mscred(df_predict_mscred_hybrid, final_threshold)
fig.show()

### Contribution

In [ ]:
def contribution_mscred_hybrid(model, df_test, timestamps, config, scaler_params, start_date, end_date):
    device = next(model.parameters()).device
    model.eval()

    if not isinstance(timestamps, pd.Index):
        timestamps = pd.to_datetime(timestamps)
        
    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)
    
    mask_period = (timestamps >= start_dt) & (timestamps <= end_dt)
    if not mask_period.any():
        raise ValueError(f"Nenhum dado encontrado entre {start_date} e {end_date}")
        
    valid_indices = np.where(mask_period)[0]
    idx_start = valid_indices[0]
    idx_end = valid_indices[-1]
    
    # Definição do Buffer para predição (Histórico)
    gap_time = config['gap_time']
    win_size = config['win_size']
    step_max = config['step_max']
    
    buffer_steps = (win_size[-1] // gap_time) + step_max + 2
    buffer_indices = buffer_steps * gap_time
    
    slice_start = max(0, idx_start - buffer_indices)
    slice_end = idx_end + 1
    
    df_subset = df_test.iloc[slice_start : slice_end]
    
    # Geração das Matrizes
    X_subset = generate_signature_matrix(df_subset, config, scaler_params)
    
    if len(X_subset) == 0:
        raise ValueError("Intervalo muito curto para gerar matrizes.")

    min_val, max_val = scaler_params
    raw_values = np.array(df_subset.values, dtype=np.float64)
    
    # Normalização
    values_norm = (raw_values.T - min_val) / (max_val - min_val + 1e-6)
    values_norm = values_norm.T
    
    # Alinhamento (MSCRED descarta o início correspondente ao buffer)
    start_gap_steps = (win_size[-1] // gap_time) + step_max
    start_index_real = start_gap_steps * gap_time
    
    y_subset = values_norm[start_index_real :: gap_time]

    min_len = min(len(X_subset), len(y_subset))
    X_subset = X_subset[:min_len].astype(np.float32)
    y_subset = y_subset[:min_len].astype(np.float32)

    # Dataloader
    dataset = HybridDataset(X_subset, y_subset)
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    
    # Matriz para acumular erro
    total_error_matrix = np.zeros((config['sensor_n'], config['sensor_n']), dtype=np.float64)
    samples_count = 0
    
    with torch.no_grad():
        for batch_x, _ in loader:
            batch_x = batch_x.to(device)
            
            # tuple (recon_matrix, recon_values)
            recon_matrix, _ = model(batch_x)
            
            recon_np = recon_matrix.cpu().numpy()
            input_np = batch_x.cpu().numpy()
            
            # Ground Truth (Último passo da sequência)
            gt_batch = input_np[:, -1, :, :, :]
            
            # Erro Quadrático
            diff = np.square(gt_batch - recon_np)
            
            # Média sobre Escalas (Batch, H, W)
            diff_aggregated_scales = np.mean(diff, axis=1)
            
            # Soma sobre o Tempo do Batch (H, W)
            batch_total_error = np.sum(diff_aggregated_scales, axis=0)
            
            total_error_matrix += batch_total_error
            samples_count += batch_x.shape[0]

    sensor_scores = np.sum(total_error_matrix, axis=1)
    total_period_error = np.sum(sensor_scores)
    
    variable_names = df_test.columns
    diagnosis_df = pd.DataFrame({
        'Variable': variable_names,
        'Total_Error_Contribution': sensor_scores,
        'Contribution %': (sensor_scores / total_period_error) * 100
    })
    
    diagnosis_df = diagnosis_df.sort_values(by='Total_Error_Contribution', ascending=False).reset_index(drop=True)
    
    return diagnosis_df

In [ ]:
start_anomaly = '2025-02-16 12:00:00'
end_anomaly = '2025-02-19 03:08:00'

df_contribution_mscred_hybrid = contribution_mscred_hybrid(
    model=model_hybrid,
    df_test=df_test,
    timestamps=eixoX_test,
    config=model_config,
    scaler_params=scaler,
    start_date=start_anomaly,
    end_date=end_anomaly
)
df_contribution_mscred_hybrid

### Reconstruction

In [ ]:
def plot_variable_reconstruction_mscred(model, df_test, timestamps, target_var_name, config, scaler_params):
    device = next(model.parameters()).device
    model.eval()

    try:
        var_idx = list(df_test.columns).index(target_var_name)
    except ValueError:
        print(f"Erro: Variável '{target_var_name}' não encontrada no DataFrame.")
        return

    # Gerando das Matrizes (X)
    X_test = generate_signature_matrix(df_test, config, scaler_params)
    
    if len(X_test) == 0:
        print("Dados insuficientes para gerar matrizes.")
        return
        
    # float32 para economizar memória
    X_test = X_test.astype(np.float32)

    min_val, max_val = scaler_params
    
    # Normalização MinMax
    raw_values = np.array(df_test.values, dtype=np.float64)
    # Transposição para (Sensores x Amostras)
    values_norm = (raw_values.T - min_val) / (max_val - min_val + 1e-6)
    values_norm = values_norm.T # Volta para (Amostras, Sensores)
    
    # Alinhamento Temporal (Corte inicial devido ao histórico do MSCRED)
    gap_time = config['gap_time']
    win_size = config['win_size']
    step_max = config['step_max']
    
    start_gap_steps = (win_size[-1] // gap_time) + step_max
    start_index_real = start_gap_steps * gap_time
    
    y_test = values_norm[start_index_real :: gap_time]
    
    min_len = min(len(X_test), len(y_test))
    X_test = X_test[:min_len]
    y_test = y_test[:min_len].astype(np.float32)

    dataset = HybridDataset(X_test, y_test)
    loader = DataLoader(dataset, batch_size=config.get('batch_size', 128), shuffle=False)
    
    original_vals_norm = []
    reconstructed_vals_norm = []
    
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            
            # shape (Matriz, Valores)
            _, recon_values = model(batch_x)
            
            original_vals_norm.extend(batch_y[:, var_idx].cpu().numpy())
            reconstructed_vals_norm.extend(recon_values[:, var_idx].cpu().numpy())
            
    # Inverse Transform (Valor_Real = Valor_Norm * (Max - Min) + Min)
    var_min = min_val[var_idx].item()
    var_max = max_val[var_idx].item()
    
    orig_real = np.array(original_vals_norm) * (var_max - var_min) + var_min
    recon_real = np.array(reconstructed_vals_norm) * (var_max - var_min) + var_min
    
    if hasattr(timestamps, 'values'):
        ts_values = timestamps.values
    else:
        ts_values = np.array(timestamps)
        
    valid_timestamps = ts_values[start_index_real :: gap_time]
    plot_len = min(len(orig_real), len(valid_timestamps))
    
    df_plot = pd.DataFrame({
        'Original': orig_real[:plot_len],
        'Reconstruido': recon_real[:plot_len]
    }, index=valid_timestamps[:plot_len])
    if not isinstance(df_plot.index, pd.DatetimeIndex):
        df_plot.index = pd.to_datetime(df_plot.index)
    
    df_resampled = df_plot.resample('1min').mean().fillna(0)
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_resampled.index, 
        y=df_resampled['Original'], 
        name='Valor Real'
    ))
    fig.add_trace(go.Scatter(
        x=df_resampled.index, 
        y=df_resampled['Reconstruido'], 
        name='Valor Reconstruído (Latente)'
    ))
    
    fig.update_layout(
        xaxis_title="Tempo",
        yaxis_title="Valor do Sensor",
        hovermode="x unified",
        template="plotly_white"
    )
    fig.show()

plot_variable_reconstruction_mscred(
    model=model_hybrid, 
    df_test=df_test,
    timestamps=eixoX_test,
    target_var_name="ARA-384V984.PV",
    config=model_config,
    scaler_params=scaler
)

In [ ]:
plot_variable_reconstruction_mscred(
    model=model_hybrid, 
    df_test=df_test,
    timestamps=eixoX_test,
    target_var_name="ARA-384I607.PV",
    config=model_config,
    scaler_params=scaler
)

### Class implementation

In [ ]:
from MSCRED import MSCRED

df_train_list = [
    df_train1,
    df_train2,
    df_train3,
    df_train4,
    df_train5,
    df_train6,
    df_train7,
    df_train8,
    df_train9,
    df_train10,
    df_train11,
    df_train12,
    df_train13,
    df_train14
]

mscred = MSCRED()

gain = 0.9
epochs = 5
mscred.fit(df_train_list, gain=gain, epochs=epochs)
predictions = mscred.predict(df_test, eixoX_test)
contribution = mscred.contribution(df_test, eixoX_test)

## Robust Unsupervised Anomaly Detection With Variational Autoencoder in Multivariate Time Series Data (MSCVAE)
Yokkampon, U., Mowshowitz, A., Chumkamon, S., & Hayashi, E. (2022). Robust unsupervised anomaly detection with variational autoencoder in multivariate time series data. IEEE Access, 10, 57835-57849.

In [ ]:
class AttributeMatrixGenerator:
    def __init__(self, window_size=10, step=10):
        # [cite_start]O paper define w=10 e intervalo=10 [cite: 153, 154]
        self.w = window_size
        self.step = step
        # Armazena estatísticas de normalização
        self.mean = None
        self.std = None

    def fit_scaler(self, train_dataframes):
        """
        Calcula média e desvio padrão global a partir de uma lista de DataFrames de treino.
        
        Args:
            train_dataframes (list of pd.DataFrame): Lista contendo [df_train1, df_train2, ...]
        """
        # Concatena todos os dfs de treino apenas para calcular as estatísticas globais
        full_train_df = pd.concat(train_dataframes, ignore_index=True)
        
        self.mean = full_train_df.mean()
        self.std = full_train_df.std() + 1e-6 # Evita divisão por zero

    def generate(self, df):
        """
        Gera as matrizes de atributos usando a normalização aprendida no fit.
        """
        if self.mean is None or self.std is None:
            raise ValueError("O gerador não foi treinado. Execute .fit_scaler([df_train1, ...]) antes de gerar matrizes.")

        # Normalização usando as estatísticas do TREINO (crucial para validação correta)
        # Alinha as colunas para garantir que estamos subtraindo a média da feature correta
        try:
            data = (df - self.mean) / self.std
        except ValueError as e:
            print(f"Erro na normalização: Verifique se as colunas do df batem com o treino. {e}")
            raise

        values = data.values
        # Se houver NaN após normalização (ex: desvio padrão zero), preenche com 0
        values = np.nan_to_num(values) 
        
        matrices = []
        
        # Janela deslizante
        # Se o dataframe for menor que a janela, retorna tensor vazio ou lida conforme necessário
        if len(values) < self.w:
            return torch.empty(0)

        for t in range(self.w, len(values), self.step):
            # Segmento X: [w, n]
            x_segment = values[t-self.w : t] 
            
            # Eq. 1: Produto interno par-a-par dividido por w 
            # (n, w) @ (w, n) -> (n, n)
            x_t = torch.tensor(x_segment, dtype=torch.float32).T
            m_t = torch.matmul(x_t, x_t.T) / self.w
            matrices.append(m_t)
            
        if not matrices:
            return torch.empty(0)

        # Formato final: (Batch, Channels=1, N, N)
        return torch.stack(matrices).unsqueeze(1)

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(TemporalAttention, self).__init__()
        self.scale = 5.0 # Chi = 5 conforme definido no paper [cite: 249]

    def forward(self, h_current, h_history):
        # h_current: (B, C, H, W) -> hidden state atual
        # h_history: Lista de hidden states passados
        if not h_history:
            return h_current
        
        # Empilha histórico: (Time, B, C, H, W)
        context = torch.stack(h_history, dim=0)
        
        # Eq. 7: Score s^t = (H^t . H^i) / chi [cite: 248]
        # Atenção baseada em produto escalar achatado
        b, c, h, w = h_current.size()
        flat_curr = h_current.view(b, -1).unsqueeze(1) # (B, 1, Features)
        flat_hist = context.view(context.size(0), b, -1).permute(1, 2, 0) # (B, Features, Time)
        
        scores = torch.bmm(flat_curr, flat_hist) / self.scale # (B, 1, Time)
        weights = F.softmax(scores, dim=-1) # Eq. 6 [cite: 247]
        
        # Eq. 5: Soma ponderada [cite: 245]
        # (B, 1, Time) @ (B, Time, Features) -> (B, 1, Features)
        weighted_hist = torch.bmm(weights, flat_hist.permute(0, 2, 1))
        h_hat = weighted_hist.view(b, c, h, w)
        
        return h_hat

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.padding = kernel_size // 2
        
        self.conv = nn.Conv2d(in_channels=input_dim + hidden_dim,
                              out_channels=4 * hidden_dim,
                              kernel_size=kernel_size,
                              padding=self.padding,
                              bias=bias)

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class MSCVAE(nn.Module):
    def __init__(self, n_features, latent_dim=round(n_features/2)):
        super(MSCVAE, self).__init__()
        self.n_features = n_features
        
        # [cite_start]--- Encoder (Convolutional) [cite: 178] ---
        # Assume input (1, N, N)
        self.enc1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)
        self.enc2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        
        with torch.no_grad():
            # Cria um tensor dummy com o shape (Batch=1, Channel=1, H=n_features, W=n_features)
            dummy_input = torch.zeros(1, 1, n_features, n_features)
            dummy_out = self.enc3(self.enc2(self.enc1(dummy_input)))
            
            # Pega o número total de features achatadas (ex: 64 * 4 * 4 = 1024)
            self.flatten_dim = dummy_out.view(1, -1).size(1)
            # Salva o shape espacial para usar no decoder (ex: 64, 4, 4)
            self.spatial_shape = dummy_out.shape[1:] 

        # Variáveis latentes (Mean & LogVar)
        self.fc_mu = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, self.flatten_dim)
        
        # Temporal Modeling (ConvLSTM + Attention)
        # Usamos ConvLSTM no bottleneck/nível mais profundo conforme Fig 1 (b)-(c)
        self.clstm = ConvLSTMCell(64, 64, kernel_size=3, bias=True)
        self.attention = TemporalAttention(hidden_dim=64)
        
        # [cite_start]--- Decoder (Deconvolutional) [cite: 254] ---
        self.dec3 = nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.dec2 = nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.dec1 = nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1)
        
        # Estado interno para LSTM e histórico para atenção
        self.h_state = None
        self.c_state = None
        self.history = []
        self.max_history = 5 # Janela de atenção (h na Eq. 5)

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            return mu + torch.randn_like(std) * std
        else:
            return mu

    def forward(self, x):
        batch_size = x.size(0)
        
        # Encoding
        e1 = F.relu(self.enc1(x))
        e2 = F.relu(self.enc2(e1))
        e3 = F.relu(self.enc3(e2)) # Feature map mais profunda (P^t,l)
        
        # VAE Bottleneck
        flat = e3.view(batch_size, -1)
        mu = self.fc_mu(flat)
        logvar = self.fc_logvar(flat)
        z = self.reparameterize(mu, logvar)
        
        # [cite_start]3. Temporal Modeling (Attention-based ConvLSTM) [cite: 242, 245]
        if self.h_state is None or self.h_state.size(0) != batch_size:
            h, w = self.spatial_shape[1], self.spatial_shape[2]
            self.h_state = torch.zeros(batch_size, 64, h, w).to(x.device)
            self.c_state = torch.zeros(batch_size, 64, h, w).to(x.device)
            self.history = []

        # Aplica ConvLSTM na feature map latente
        h_next, c_next = self.clstm(e3, (self.h_state, self.c_state))
        
        # Aplica Atenção no histórico
        h_att = self.attention(h_next, self.history)
        
        # Atualiza estados
        self.h_state = h_att.detach() 
        self.c_state = c_next.detach()
        self.history.append(self.h_state)
        if len(self.history) > self.max_history:
            self.history.pop(0)

        # [cite_start]4. Decoding [cite: 262]
        z_dec = self.fc_decode(z)
        # Reshape usando as dimensões espaciais calculadas dinamicamente
        z_dec = z_dec.view(batch_size, *self.spatial_shape)
        
        combined_features = z_dec + h_att 
        
        d3 = F.relu(self.dec3(combined_features))
        d2 = F.relu(self.dec2(d3))
        recon_x = self.dec1(d2)
        
        if recon_x.shape != x.shape:
            recon_x = F.interpolate(recon_x, size=x.shape[2:], mode='bilinear', align_corners=False)

        return recon_x, mu, logvar

    def loss_function(self, recon_x, x, mu, logvar):
        # Eq. [cite_start]2 e 9: MSE + KL Divergence [cite: 173, 268]
        MSE = F.mse_loss(recon_x, x, reduction='sum')
        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        return MSE + KLD

def train_model(model, train_loader, epochs=50):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    
    for epoch in range(epochs):
        epoch=epoch+1
        total_loss = 0
        for batch in train_loader:
            x = batch[0]
            
            optimizer.zero_grad()
            recon_x, mu, logvar = model(x)
            loss = model.loss_function(recon_x, x, mu, logvar)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Epoch {epoch} | Loss: {total_loss / len(train_loader.dataset):.4f}")

def get_anomaly_scores(model, data_tensor):
    model.eval()
    mse_loss = nn.MSELoss(reduction='none')
    scores = []
    
    with torch.no_grad():
        model.h_state = None
        model.history = []
        
        # Processar sequencialmente para manter memória temporal
        for i in range(data_tensor.size(0)):
            x = data_tensor[i].unsqueeze(0)
            recon_x, _, _ = model(x)
            # Score é o erro de reconstrução
            loss = mse_loss(recon_x, x).sum().item()
            scores.append(loss)
            
    return np.array(scores)

def calculate_threshold(scores, gain=3):    
    mu = np.mean(scores)
    sigma = np.std(scores)
    threshold = mu + (gain * sigma)
    
    print(f"Threshold: {threshold:.4f}")
    return threshold

### Fit

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WINDOW_SIZE = 10
STRIDE = 1
BATCH_SIZE = 128
N_FEATURES = df_train.shape[1]
EPOCHS = 15

generator = AttributeMatrixGenerator(window_size=WINDOW_SIZE, step=STRIDE)
dfs_treino = [df_train1, df_train2, df_train3, df_train4, df_train5, df_train6, df_train7, df_train8, df_train9, df_train10, df_train11, df_train12, df_train13, df_train14]
generator.fit_scaler(dfs_treino)

train_tensors = []
for df in dfs_treino:
    t = generator.generate(df)
    if t.nelement() > 0:
        train_tensors.append(t)
final_train_tensor = torch.cat(train_tensors, dim=0)
train_loader = DataLoader(TensorDataset(final_train_tensor), batch_size=BATCH_SIZE, shuffle=True)

# Instanciar modelo
n_vars = N_FEATURES
model_mscvae = MSCVAE(n_features=n_vars)

# Treino
train_model(model_mscvae, train_loader,epochs=EPOCHS)
train_scores = get_anomaly_scores(model_mscvae, final_train_tensor)
threshold = calculate_threshold(train_scores, gain=5)

### Predict

In [ ]:
# Predict
def predict_mscvae(df_test, timestamps, model, generator, batch_size=128, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model.to(device)
    model.eval()
    
    tensor_test = generator.generate(df_test)
    loader = DataLoader(TensorDataset(tensor_test), batch_size=batch_size, shuffle=False)
    
    mse_loss = nn.MSELoss(reduction='none')
    all_scores = []
    
    # Inferência
    with torch.no_grad():
        model.h_state = None
        model.c_state = None
        model.history = []
        
        for (batch_input,) in loader:
            inputs = batch_input.to(device)
            
            # Forward pass
            recon_x, mu, logvar = model(inputs)
            
            # Cálculo do Score: MSE por amostra
            # inputs shape: (B, 1, N, N)
            # sum((recon - input)^2) -> reduz dimensões (1, 2, 3) mantendo o batch
            loss_batch = mse_loss(recon_x, inputs).sum(dim=(1, 2, 3))
            
            all_scores.extend(loss_batch.cpu().numpy())

    # Alinhamento Temporal
    # O generator produz matrizes começando no índice 'window_size' e pulando a cada 'step'
    w = generator.w
    s = generator.step
    

    if hasattr(timestamps, 'values'):
        ts_values = timestamps.values
    else:
        ts_values = np.array(timestamps)

    valid_indices = range(w, len(df_test) + 1, s) # +1 para incluir o último ponto se divisível exato
    min_len = min(len(all_scores), len(valid_indices))
    
    final_scores = all_scores[:min_len]
    final_indices = valid_indices[:min_len]
    
    final_timestamps = []
    for idx in final_indices:
        if idx < len(ts_values):
             final_timestamps.append(ts_values[idx])
        elif idx == len(ts_values):
            final_timestamps.append(ts_values[-1])
    final_scores = final_scores[:len(final_timestamps)]
    
    # DataFrame
    df_predict = pd.DataFrame({
        'timestamp': final_timestamps,
        'loss': final_scores
    })
    df_predict.set_index('timestamp', inplace=True)
    df_predict = df_predict.asfreq('1min')
    df_predict['loss'] = df_predict['loss'].fillna(0)
    
    return df_predict

df_predict_mscvae = predict_mscvae(
    df_test=df_test,            # Seu DataFrame de teste
    timestamps=eixoX_test,      # Série de timestamps
    model=model_mscvae,         # Modelo treinado
    generator=generator,        # Instância do generator
    batch_size=BATCH_SIZE,
    device=DEVICE
)

In [ ]:
def plot_predict(df_predict, threshold):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_predict.index,
        y=df_predict['loss'],
        mode='lines',
        name='Erro de Reconstrução',
        fill="tozeroy", 
        line=dict(color="#0F293A"),
        connectgaps=False
    ))
    
    fig.add_hline(
        y=threshold, 
        name='Limiar',
        line_dash="dash", 
        line_color="red", 
        line_width=2
    )
    
    fig.update_layout(
        xaxis_title='Data/Hora',
        yaxis_title='Erro Médio Absoluto (MAE)',
        template='plotly_white',
        hovermode="x unified",
        showlegend=True
    )

    return fig

fig = plot_predict(df_predict_mscvae, threshold)
fig.show()

### Contribution

In [ ]:
def contribution(model, generator, df_test, timestamps, start_date, end_date, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
    model.to(device)
    model.eval()
    
    # Converter para datetime para garantir comparação correta
    if not np.issubdtype(timestamps.dtype, np.datetime64):
        timestamps = pd.to_datetime(timestamps)
    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)
    
    # 1. Encontrar índices correspondentes nas datas externas
    # Busca booleana para encontrar o intervalo
    mask = (timestamps >= start_dt) & (timestamps <= end_dt)
    
    if not mask.any():
        raise ValueError(f"Nenhuma data encontrada no intervalo {start_date} a {end_date}")
        
    # Pega o primeiro e o último índice onde a condição é verdadeira
    valid_indices = np.where(mask)[0]
    idx_start = valid_indices[0]
    idx_end = valid_indices[-1]
    
    # 2. Recorte com Margem (Window Size)
    w = generator.w
    
    # O recorte deve começar 'w' passos antes para que a primeira janela gerada
    # corresponda exatamente ao tempo 'idx_start'.
    slice_start = max(0, idx_start - w)
    slice_end = idx_end + 1
    
    # Recorta o df_test original usando os índices numéricos
    subset_df = df_test.iloc[slice_start : slice_end]
    
    if len(subset_df) < w:
        raise ValueError("Intervalo selecionado (com margem) é menor que o tamanho da janela.")

    # 3. Geração das Matrizes
    # O generator retornará tensor (T, 1, N, N)
    # Nota: Se o subset for exatamente o tamanho da janela, retorna 1 matriz.
    range_tensor = generator.generate(subset_df)
    
    if range_tensor.nelement() == 0:
        raise ValueError("Nenhuma matriz gerada. Verifique se o intervalo é suficiente para o window/step.")

    loader = DataLoader(TensorDataset(range_tensor), batch_size=32, shuffle=False)
    
    n_features = range_tensor.shape[2]
    total_error_matrix = torch.zeros(n_features, n_features).to(device)
    
    # 4. Inferência e Acumulação
    with torch.no_grad():
        model.h_state = None
        model.c_state = None
        model.history = []
        
        for (batch_input,) in loader:
            inputs = batch_input.to(device)
            recon_x, _, _ = model(inputs)
            
            # Erro quadrático: (B, N, N)
            batch_error = torch.pow(inputs - recon_x, 2).squeeze(1)
            
            # Acumula erro
            total_error_matrix += torch.sum(batch_error, dim=0)
            
    # 5. Relatório
    variable_scores = torch.sum(total_error_matrix, dim=1).cpu().numpy()
    variable_names = df_test.columns
    total_period_error = np.sum(variable_scores)
    
    diagnosis_df = pd.DataFrame({
        'Variable': variable_names,
        'Total_Error_Contribution': variable_scores,
        'Contribution %': (variable_scores / total_period_error) * 100
    })
    
    diagnosis_df = diagnosis_df.sort_values(by='Total_Error_Contribution', ascending=False).reset_index(drop=True)
    
    return diagnosis_df

In [ ]:
## Definir o intervalo da anomalia
start_anomaly = '2025-02-15 23:47:00'
end_anomaly = '2025-02-19 03:08:00'

# start_anomaly = '2025-02-05 01:50:00'
# end_anomaly = '2025-02-06 14:08:00'

# start_anomaly = '2025-02-23 03:50:00'
# end_anomaly = '2025-02-27 02:00:00'

range_diagnosis = contribution(
    model=model_mscvae,
    generator=generator,
    df_test=df_test,
    timestamps=eixoX_test,
    start_date=start_anomaly,
    end_date=end_anomaly,
    device=DEVICE
)
range_diagnosis

## Robust Unsupervised Anomaly Detection With Variational Autoencoder in Multivariate Time Series Data (Hybrid MSCVAE) [Adaptação para recuperar reconstrução das variáveis]
Yokkampon, U., Mowshowitz, A., Chumkamon, S., & Hayashi, E. (2022). Robust unsupervised anomaly detection with variational autoencoder in multivariate time series data. IEEE Access, 10, 57835-57849.

In [ ]:
class AttributeMatrixGenerator:
    def __init__(self, window_size=10, step=10):
        self.w = window_size
        self.step = step
        self.mean = None
        self.std = None

    def fit_scaler(self, train_dataframes):
        # Concatena para calcular estatísticas globais
        full_train_df = pd.concat(train_dataframes, ignore_index=True)
        self.mean = full_train_df.mean()
        self.std = full_train_df.std() + 1e-6

    def generate(self, df):
        if self.mean is None:
            raise ValueError("Execute .fit_scaler() primeiro.")

        # Normalização
        data = (df - self.mean) / self.std
        values = np.nan_to_num(data.values)
        
        matrices = []
        target_values = [] 

        if len(values) < self.w:
            # Retorna tupla vazia se df for muito pequeno (i.e menor que window_size)
            return torch.empty(0), torch.empty(0)

        for t in range(self.w, len(values), self.step):
            x_segment = values[t-self.w : t] 
            
            # Matriz (Eq. 1)
            x_t = torch.tensor(x_segment, dtype=torch.float32).T
            m_t = torch.matmul(x_t, x_t.T) / self.w
            matrices.append(m_t)
            
            last_val = torch.tensor(x_segment[-1], dtype=torch.float32)
            target_values.append(last_val)
            
        if not matrices:
            return torch.empty(0), torch.empty(0)

        # Tuple: (Tensor das Matrizes, Tensor dos Valores)
        return torch.stack(matrices).unsqueeze(1), torch.stack(target_values)

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(TemporalAttention, self).__init__()
        self.scale = 5.0 # Mecanismo de atenção (quantas janelas o algoritmo olha para realizar a predição atual)

    def forward(self, h_current, h_history):
        if not h_history:
            return h_current
        
        context = torch.stack(h_history, dim=0)
        b, c, h, w = h_current.size()
        flat_curr = h_current.view(b, -1).unsqueeze(1) 
        flat_hist = context.view(context.size(0), b, -1).permute(1, 2, 0) 
        
        scores = torch.bmm(flat_curr, flat_hist) / self.scale 
        weights = F.softmax(scores, dim=-1) 
        
        weighted_hist = torch.bmm(weights, flat_hist.permute(0, 2, 1))
        h_hat = weighted_hist.view(b, c, h, w)
        
        return h_hat

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.padding = kernel_size // 2
        
        self.conv = nn.Conv2d(in_channels=input_dim + hidden_dim,
                              out_channels=4 * hidden_dim,
                              kernel_size=kernel_size,
                              padding=self.padding,
                              bias=bias)

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class MSCVAE_Hybrid(nn.Module):
    def __init__(self, n_features):
        super(MSCVAE_Hybrid, self).__init__()
        self.n_features = n_features
        latent_dim=round(math.sqrt(n_features))
        
        # Encoder
        self.enc1 = nn.Conv2d(1, 16, 3, 2, 1)
        self.enc2 = nn.Conv2d(16, 32, 3, 2, 1)
        self.enc3 = nn.Conv2d(32, 64, 3, 2, 1)
        
        # Dummy pass
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_features, n_features)
            out = self.enc3(self.enc2(self.enc1(dummy)))
            self.flatten_dim = out.view(1, -1).size(1)
            self.spatial_shape = out.shape[1:] 

        # Latente
        self.fc_mu = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_dim, latent_dim)
        
        # Decoder Matriz
        self.fc_decode_mat = nn.Linear(latent_dim, self.flatten_dim)
        
        # Decoder de Valores (Híbrido)
        self.val_decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, n_features) 
        )

        # Temporal Modeling
        self.clstm = ConvLSTMCell(64, 64, kernel_size=3, bias=True)
        self.attention = TemporalAttention(hidden_dim=64)
        
        # Decoder Convolucional
        self.dec3 = nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1)
        self.dec2 = nn.ConvTranspose2d(32, 16, 3, 2, 1, output_padding=1)
        self.dec1 = nn.ConvTranspose2d(16, 1, 3, 2, 1, output_padding=1)
        
        self.h_state = None; self.c_state = None; self.history = []; self.max_history = 5

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            return mu + torch.randn_like(std) * std
        else:
            return mu

    def forward(self, x):
        batch_size = x.size(0)
        
        # Encoder
        e3 = F.relu(self.enc3(F.relu(self.enc2(F.relu(self.enc1(x))))))
        flat = e3.view(batch_size, -1)
        mu = self.fc_mu(flat)
        logvar = self.fc_logvar(flat)
        z = self.reparameterize(mu, logvar)
        
        # Rota 1: Valores
        recon_values = self.val_decoder(z)
        
        # Rota 2: Matrizes
        if self.h_state is None or self.h_state.size(0) != batch_size:
            h, w = self.spatial_shape[1], self.spatial_shape[2]
            self.h_state = torch.zeros(batch_size, 64, h, w).to(x.device)
            self.c_state = torch.zeros(batch_size, 64, h, w).to(x.device)
            self.history = []

        h_next, c_next = self.clstm(e3, (self.h_state, self.c_state))
        h_att = self.attention(h_next, self.history)
        
        self.h_state = h_att.detach()
        self.c_state = c_next.detach()
        self.history.append(self.h_state)
        if len(self.history) > self.max_history: self.history.pop(0)

        z_dec = self.fc_decode_mat(z).view(batch_size, *self.spatial_shape)
        combined = z_dec + h_att
        
        d3 = F.relu(self.dec3(combined))
        d2 = F.relu(self.dec2(d3))
        recon_matrix = self.dec1(d2)
        
        if recon_matrix.shape != x.shape:
            recon_matrix = F.interpolate(recon_matrix, size=x.shape[2:], mode='bilinear', align_corners=False)

        return recon_matrix, recon_values, mu, logvar

    def loss_function(self, recon_matrix, x_matrix, recon_values, x_values, mu, logvar, alpha=1):
        MSE_Mat = F.mse_loss(recon_matrix, x_matrix, reduction='sum')
        MSE_Val = F.mse_loss(recon_values, x_values, reduction='sum')
        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        
        return MSE_Mat + (alpha * MSE_Val) + KLD

def train_hybrid_model(model, train_loader, epochs=50):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        for batch_matrix, batch_values in train_loader:
            x_mat = batch_matrix.to(DEVICE)
            x_val = batch_values.to(DEVICE)
            
            optimizer.zero_grad()
            
            # Forward
            recon_mat, recon_val, mu, logvar = model(x_mat)
            
            # Loss
            loss = model.loss_function(recon_mat, x_mat, recon_val, x_val, mu, logvar, alpha=1)
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_loader.dataset):.4f}")

def get_anomaly_scores(model, data_tensor):
    model.eval()
    mse_loss = nn.MSELoss(reduction='none')
    scores = []
    
    with torch.no_grad():
        model.h_state = None
        model.history = []
        
        for i in range(data_tensor.size(0)):
            x = data_tensor[i].unsqueeze(0).to(DEVICE)
        
            recon_mat, _, _, _ = model(x)
            
            # Score baseado apenas na reconstrução da matriz
            loss = mse_loss(recon_mat, x).sum().item()
            scores.append(loss)
            
    return np.array(scores)

def calculate_threshold(scores, gain=3):    
    mu = np.mean(scores)
    sigma = np.std(scores)
    threshold = mu + (gain * sigma)
    print(f"Threshold: {threshold:.4f}")
    return threshold

### Fit

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WINDOW_SIZE = 10
STRIDE = 1
BATCH_SIZE = 128

dfs_treino = [df_train1, df_train2, df_train3, df_train4, df_train5, df_train6, df_train7, df_train8, df_train9, df_train10, df_train11, df_train12, df_train13, df_train14]
N_FEATURES = dfs_treino[0].shape[1]
EPOCHS = 15

generator = AttributeMatrixGenerator(window_size=WINDOW_SIZE, step=STRIDE)
generator.fit_scaler(dfs_treino)

train_matrices = []
train_values = []

for df in dfs_treino:
    t_mat, t_val = generator.generate(df)
    if t_mat.nelement() > 0:
        train_matrices.append(t_mat)
        train_values.append(t_val)

final_train_matrix = torch.cat(train_matrices, dim=0)
final_train_values = torch.cat(train_values, dim=0)

train_loader = DataLoader(
    TensorDataset(final_train_matrix, final_train_values), 
    batch_size=BATCH_SIZE, 
    shuffle=True
)
model_hybrid = MSCVAE_Hybrid(n_features=N_FEATURES).to(DEVICE)
train_hybrid_model(model_hybrid, train_loader, epochs=EPOCHS)

train_scores = get_anomaly_scores(model_hybrid, final_train_matrix)
# threshold = calculate_threshold(train_scores, gain=3)

In [ ]:
class SPOT:
    def __init__(self, q=1e-4):
        self.proba = q
        self.extreme_quantile = None
        self.data = None
        self.init_data = None
        self.init_threshold = None
        self.peaks = None
        self.n = 0
        self.Nt = 0

    def fit(self, init_data, data):
        if isinstance(data, list): self.data = np.array(data)
        elif isinstance(data, np.ndarray): self.data = data
        elif isinstance(data, pd.Series): self.data = data.values
        else: return
        if isinstance(init_data, list): self.init_data = np.array(init_data)
        elif isinstance(init_data, np.ndarray): self.init_data = init_data
        elif isinstance(init_data, pd.Series): self.init_data = init_data.values
        elif isinstance(init_data, int):
            self.init_data = self.data[:init_data]
            self.data = self.data[init_data:]
        elif isinstance(init_data, float) and (init_data < 1) and (init_data > 0):
            r = int(init_data * data.size)
            self.init_data = self.data[:r]
            self.data = self.data[r:]
        else: return

    def add(self, data):
        if isinstance(data, list): data = np.array(data)
        elif isinstance(data, np.ndarray): data = data
        elif isinstance(data, pd.Series): data = data.values
        else: return
        self.data = np.append(self.data, data)

    def initialize(self, level=0.98, min_extrema=False, verbose=True):
        if min_extrema:
            self.init_data = -self.init_data
            self.data = -self.data
            level = 1 - level
        level = level - math.floor(level)
        n_init = self.init_data.size
        S = np.sort(self.init_data)
        self.init_threshold = S[int(level * n_init)]
        self.peaks = self.init_data[self.init_data > self.init_threshold] - self.init_threshold
        self.Nt = self.peaks.size
        self.n = n_init
        if verbose:
            print('Initial threshold : %s' % self.init_threshold)
            print('Number of peaks : %s' % self.Nt)
            print('Grimshaw maximum log-likelihood estimation ... ', end='')
        g, s, l = self._grimshaw()
        self.extreme_quantile = self._quantile(g, s)
        if verbose:
            print('Extreme quantile (probability = %s): %s' % (self.proba, self.extreme_quantile))

    def _rootsFinder(fun, jac, bounds, npoints, method):
        from scipy.optimize import minimize
        if method == 'regular':
            step = (bounds[1] - bounds[0]) / (npoints + 1)
            if step == 0: bounds, step = (0, 1e-4), 1e-5
            X0 = np.arange(bounds[0] + step, bounds[1], step)
        elif method == 'random':
            X0 = np.random.uniform(bounds[0], bounds[1], npoints)
        def objFun(X, f, jac):
            g = 0
            j = np.zeros(X.shape)
            i = 0
            for x in X:
                fx = f(x)
                g = g + fx ** 2
                j[i] = 2 * fx * jac(x)
                i = i + 1
            return g, j
        opt = minimize(lambda X: objFun(X, fun, jac), X0,
                       method='L-BFGS-B',
                       jac=True, bounds=[bounds] * len(X0))
        X = opt.x
        np.round(X, decimals=5)
        return np.unique(X)

    def _log_likelihood(Y, gamma, sigma):
        n = Y.size
        if gamma != 0:
            tau = gamma / sigma
            L = -n * math.log(sigma) - (1 + (1 / gamma)) * (np.log(1 + tau * Y)).sum()
        else:
            L = n * (1 + math.log(Y.mean()))
        return L

    def _grimshaw(self, epsilon=1e-8, n_points=10):
        def u(s): return 1 + np.log(s).mean()
        def v(s): return np.mean(1 / s)
        def w(Y, t):
            s = 1 + t * Y
            us = u(s); vs = v(s)
            return us * vs - 1
        def jac_w(Y, t):
            s = 1 + t * Y
            us = u(s); vs = v(s)
            jac_us = (1 / t) * (1 - vs)
            jac_vs = (1 / t) * (-vs + np.mean(1 / s ** 2))
            return us * jac_vs + vs * jac_us
        Ym = self.peaks.min()
        YM = self.peaks.max()
        Ymean = self.peaks.mean()
        a = -1 / YM
        if abs(a) < 2 * epsilon: epsilon = abs(a) / n_points
        a = a + epsilon
        b = 2 * (Ymean - Ym) / (Ymean * Ym)
        c = 2 * (Ymean - Ym) / (Ym ** 2)
        left_zeros = SPOT._rootsFinder(lambda t: w(self.peaks, t),
                                       lambda t: jac_w(self.peaks, t),
                                       (a + epsilon, -epsilon),
                                       n_points, 'regular')
        right_zeros = SPOT._rootsFinder(lambda t: w(self.peaks, t),
                                        lambda t: jac_w(self.peaks, t),
                                        (b, c),
                                        n_points, 'regular')
        zeros = np.concatenate((left_zeros, right_zeros))
        gamma_best = 0
        sigma_best = Ymean
        ll_best = SPOT._log_likelihood(self.peaks, gamma_best, sigma_best)
        for z in zeros:
            gamma = u(1 + z * self.peaks) - 1
            sigma = gamma / z
            ll = SPOT._log_likelihood(self.peaks, gamma, sigma)
            if ll > ll_best:
                gamma_best = gamma
                sigma_best = sigma
                ll_best = ll
        return gamma_best, sigma_best, ll_best

    def _quantile(self, gamma, sigma):
        r = self.n * self.proba / self.Nt
        if gamma != 0: return self.init_threshold + (sigma / gamma) * (pow(r, -gamma) - 1)
        else: return self.init_threshold - sigma * math.log(r)

    def run(self, with_alarm=True, dynamic=True):
        if self.n > self.init_data.size:
            print('Warning : the algorithm seems to have already been run, you should initialize before running again')
            return {}
        th = []
        alarm = []
        for i in range(self.data.size):
            if not dynamic:
                if self.data[i] > self.init_threshold and with_alarm:
                    self.extreme_quantile = self.init_threshold
                    alarm.append(i)
            else:
                if self.data[i] > self.extreme_quantile:
                    if with_alarm: alarm.append(i)
                    else:
                        self.peaks = np.append(self.peaks, self.data[i] - self.init_threshold)
                        self.Nt += 1
                        self.n += 1
                        g, s, l = self._grimshaw()
                        self.extreme_quantile = self._quantile(g, s)
                elif self.data[i] > self.init_threshold:
                    self.peaks = np.append(self.peaks, self.data[i] - self.init_threshold)
                    self.Nt += 1
                    self.n += 1
                    g, s, l = self._grimshaw()
                    self.extreme_quantile = self._quantile(g, s)
                else:
                    self.n += 1
            th.append(self.extreme_quantile)
        return {'thresholds': th, 'alarms': alarm}

def pot_eval(init_score, q=1e-4, level=0.02):
    """
    Calcula o limiar POT estático baseado apenas nos scores de treino.
    """
    lms = level
    while True:
        try:
            s = SPOT(q)
            s.fit(init_score, init_score)
            # Initialize realiza o cálculo pesado (Grimshaw) e define o limiar
            s.initialize(level=lms, min_extrema=False, verbose=False)
        except Exception:
            # Se o Grimshaw falhar em convergir,o nível é reduzido
            lms = lms * 0.999
        else:
            break
    return s.extreme_quantile

threshold = pot_eval(train_scores, q=1e-4, level=0.02)

# Aplicação do Ganho
# Ganho é um multiplicador empírico para reduzir falsos positivos
gain = 1
final_threshold = threshold * gain

print(f"\n--- Resultados POT ---")
print(f"Limiar Base (POT): {threshold:.6f}")
print(f"Ganho Aplicado: {gain}")
print(f"Limiar Final: {final_threshold:.6f}")

### Predict

In [ ]:
def predict_mscvae_hybrid(df_test, timestamps, model, generator, batch_size=128, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model.to(device)
    model.eval()
    
    # 1. Geração de Dados (Adaptação para Tupla)
    # O generator agora retorna (Matrizes, Valores)
    try:
        tensor_matrices, tensor_values = generator.generate(df_test)
    except ValueError as e:
        print(f"Erro na geração: {e}")
        return pd.DataFrame()

    # Verificação de segurança
    if tensor_matrices.nelement() == 0:
        raise ValueError(f"DataFrame de teste muito pequeno para a janela {generator.w}.")

    # Para predição do SCORE de anomalia, usamos apenas a reconstrução da matriz.
    # Criamos o DataLoader apenas com as matrizes de entrada.
    loader = DataLoader(TensorDataset(tensor_matrices), batch_size=batch_size, shuffle=False)
    
    mse_loss = nn.MSELoss(reduction='none')
    all_scores = []
    
    # 2. Inferência
    with torch.no_grad():
        # Resetar estados da LSTM
        model.h_state = None
        model.c_state = None
        model.history = []
        
        for (batch_input,) in loader:
            inputs = batch_input.to(device) # Shape: (B, 1, N, N)
            
            # Forward pass (Adaptação para 4 saídas)
            # Retorna: recon_matrix, recon_values, mu, logvar
            # Estamos interessados apenas na recon_matrix para o score de anomalia
            recon_matrix, _, _, _ = model(inputs)
            
            # Cálculo do Score: MSE da Matriz
            # sum((recon - input)^2) -> reduz dimensões (1, 2, 3) mantendo o batch
            loss_batch = mse_loss(recon_matrix, inputs).sum(dim=(1, 2, 3))
            
            all_scores.extend(loss_batch.cpu().numpy())

    # 3. Alinhamento Temporal
    w = generator.w
    s = generator.step
    
    if hasattr(timestamps, 'values'):
        ts_values = timestamps.values
    else:
        ts_values = np.array(timestamps)

    # Lógica de Índices (Robustez)
    valid_indices = range(w, len(df_test) + 1, s)
    
    # Garante que os tamanhos batem (truncando pelo menor)
    min_len = min(len(all_scores), len(valid_indices))
    
    final_scores = all_scores[:min_len]
    final_indices = valid_indices[:min_len]
    
    final_timestamps = []
    # Proteção extra contra índice fora do array de timestamps
    for idx in final_indices:
        if idx < len(ts_values):
             final_timestamps.append(ts_values[idx])
        elif idx == len(ts_values):
            final_timestamps.append(ts_values[-1])
            
    # Ajuste final caso o loop de timestamps tenha filtrado algo
    final_scores = final_scores[:len(final_timestamps)]
    
    # 4. DataFrame Final
    df_predict = pd.DataFrame({
        'timestamp': final_timestamps,
        'loss': final_scores
    })
    df_predict.set_index('timestamp', inplace=True)
    df_predict = df_predict.asfreq('1min')
    df_predict['loss'] = df_predict['loss'].fillna(0)
    
    return df_predict

df_predict_mscvae_hybrid = predict_mscvae_hybrid(
    df_test=df_test,
    timestamps=eixoX_test,
    model=model_hybrid,
    generator=generator,
    batch_size=BATCH_SIZE,
    device=DEVICE
)

In [ ]:
def plot_predict(df_predict, threshold):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_predict.index,
        y=df_predict['loss'],
        mode='lines',
        name='Erro de Reconstrução',
        fill="tozeroy", 
        line=dict(color="#0F293A"),
        connectgaps=False
    ))
    
    fig.add_hline(
        y=threshold, 
        name='Limiar',
        line_dash="dash", 
        line_color="red", 
        line_width=2
    )
    
    fig.update_layout(
        xaxis_title='Data/Hora',
        yaxis_title='Erro Médio Absoluto (MAE)',
        template='plotly_white',
        hovermode="x unified",
        showlegend=True
    )

    return fig

fig = plot_predict(df_predict_mscvae_hybrid, final_threshold)
fig.show()

### Contribution

In [ ]:
def contribution_hybrid(model, generator, df_test, timestamps, start_date, end_date, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
    model.to(device)
    model.eval()
    
    if hasattr(timestamps, 'values'):
        ts_values = timestamps.values
    else:
        ts_values = np.array(timestamps)
        
    if not np.issubdtype(ts_values.dtype, np.datetime64):
        ts_values = pd.to_datetime(ts_values)
        
    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)
    
    mask = (ts_values >= start_dt) & (ts_values <= end_dt)
    
    if not mask.any():
        raise ValueError(f"Nenhuma data encontrada no intervalo {start_date} a {end_date}")
        
    valid_indices = np.where(mask)[0]
    idx_start = valid_indices[0]
    idx_end = valid_indices[-1]
    
    w = generator.w
    slice_start = max(0, idx_start - w)
    slice_end = idx_end + 1
    
    subset_df = df_test.iloc[slice_start : slice_end]
    
    if len(subset_df) < w:
        raise ValueError("Intervalo selecionado (com margem) é menor que o tamanho da janela.")

    try:
        tensor_matrices, _ = generator.generate(subset_df)
    except ValueError as e:
        print(f"Erro no gerador: {e}")
        return pd.DataFrame()
    
    if tensor_matrices.nelement() == 0:
        raise ValueError("Nenhuma matriz gerada. Verifique se o intervalo é suficiente.")

    loader = DataLoader(TensorDataset(tensor_matrices), batch_size=32, shuffle=False)
    
    n_features = tensor_matrices.shape[2]
    total_error_matrix = torch.zeros(n_features, n_features).to(device)
    
    with torch.no_grad():
        model.h_state = None
        model.c_state = None
        model.history = []
        
        for (batch_input,) in loader:
            inputs = batch_input.to(device) # Shape (B, 1, N, N)
            
            recon_matrix, _, _, _ = model(inputs)
            
            # Erro quadrático da matriz: (B, N, N) squeeze(1) remove o canal 1
            batch_error = torch.pow(inputs - recon_matrix, 2).squeeze(1)
            
            # Acumula erro ao longo do tempo (dimensão 0 do batch)
            total_error_matrix += torch.sum(batch_error, dim=0)
            
    variable_scores = torch.sum(total_error_matrix, dim=1).cpu().numpy()
    variable_names = df_test.columns
    total_period_error = np.sum(variable_scores)
    
    diagnosis_df = pd.DataFrame({
        'Variable': variable_names,
        'Total_Error_Contribution': variable_scores,
        'Contribution %': (variable_scores / total_period_error) * 100
    })
    
    diagnosis_df = diagnosis_df.sort_values(by='Total_Error_Contribution', ascending=False).reset_index(drop=True)
    
    return diagnosis_df

In [ ]:
## Anomalia de aumento de vibração no GARO
start_anomaly = '2025-02-15 23:47:00'
end_anomaly = '2025-02-19 03:08:00'

# start_anomaly = '2025-03-09 08:45:00'
# end_anomaly = '2025-03-09 19:10:00'

df_contribution_mscvae = contribution_hybrid(
    model=model_hybrid,
    generator=generator,
    df_test=df_test,
    timestamps=eixoX_test,
    start_date=start_anomaly,
    end_date=end_anomaly,
    device=DEVICE
)
df_contribution_mscvae

### Reconstruction

In [ ]:
def plot_variable_reconstruction(model, generator, df_test, timestamps, target_var_name, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Geração dos dados (Matrizes e Valores)
    try:
        t_mat, t_val = generator.generate(df_test)
    except ValueError as e:
        print(f"Erro no gerador: {e}")
        return

    if t_mat.nelement() == 0:
        print("Dados insuficientes para gerar janelas.")
        return

    loader = DataLoader(TensorDataset(t_mat, t_val), batch_size=128, shuffle=False)
    
    original_vals = []
    reconstructed_vals = []
    
    model.to(device)
    model.eval()
    with torch.no_grad():
        # Reset de estados
        model.h_state = None; model.c_state = None; model.history = []
        
        for x_mat, x_val in loader:
            x_mat = x_mat.to(device)
            _, recon_v, _, _ = model(x_mat)
            
            original_vals.extend(x_val.cpu().numpy())
            reconstructed_vals.extend(recon_v.cpu().numpy())
            
    # Recupera índice da variável
    try:
        idx_map = list(df_test.columns).index(target_var_name)
    except ValueError:
        print(f"Variável {target_var_name} não encontrada.")
        return

    # Recupera arrays e Desnormaliza
    orig_series = np.array(original_vals)[:, idx_map]
    recon_series = np.array(reconstructed_vals)[:, idx_map]
    
    mean = generator.mean[target_var_name]
    std = generator.std[target_var_name]
    
    orig_real = (orig_series * std) + mean
    recon_real = (recon_series * std) + mean
    
    # Alinhamento Temporal
    w = generator.w
    s = generator.step
    
    if hasattr(timestamps, 'values'):
        ts_values = timestamps.values
    else:
        ts_values = np.array(timestamps)
    
    valid_indices = [t - 1 for t in range(w, len(df_test) + 1, s)]
    
    min_len = min(len(orig_real), len(valid_indices))
    
    orig_real = orig_real[:min_len]
    recon_real = recon_real[:min_len]
    valid_indices = valid_indices[:min_len]
    
    aligned_timestamps = []
    for idx in valid_indices:
        if idx < len(ts_values):
            aligned_timestamps.append(ts_values[idx])
        else:
            if len(aligned_timestamps) > 0:
                aligned_timestamps.append(aligned_timestamps[-1])

    df_plot = pd.DataFrame({
        'Original': orig_real,
        'Reconstruido': recon_real
    }, index=pd.to_datetime(aligned_timestamps))
    
    df_plot = df_plot[~df_plot.index.duplicated(keep='first')]
    df_resampled = df_plot.resample('1min').mean().fillna(0)
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_resampled.index, 
        y=df_resampled['Original'], 
        name='Original'
    ))
    fig.add_trace(go.Scatter(
        x=df_resampled.index, 
        y=df_resampled['Reconstruido'], 
        name='Reconstruído (Via Latente)'
    ))
    
    fig.update_layout(
        xaxis_title="Tempo",
        yaxis_title="Valor",
        hovermode="x unified",
        template="plotly_white"
    )
    fig.show()

plot_variable_reconstruction(
    model=model_hybrid, 
    generator=generator, 
    df_test=df_test,
    timestamps=eixoX_test,
    target_var_name="ARA-384V984.PV", 
    device=None
)

In [ ]:
plot_variable_reconstruction(
    model=model_hybrid, 
    generator=generator, 
    df_test=df_test,
    timestamps=eixoX_test,
    target_var_name="ARA-384P930.PV", 
    device=None
)

In [ ]:
plot_variable_reconstruction(
    model=model_hybrid, 
    generator=generator, 
    df_test=df_test,
    timestamps=eixoX_test,
    target_var_name="ARA-384F920.OP", 
    device=None
)

In [ ]:
plot_variable_reconstruction(
    model=model_hybrid, 
    generator=generator, 
    df_test=df_test,
    timestamps=eixoX_test,
    target_var_name="ARA-384T671B.PV", 
    device=None
)